# NISS Statistical Report - One Click Notebook (Yearly and Monthly)

**Purpose.** Builds the NISS commercial auto statistical extract (policy transactions plus claim transactions) and writes the fixed layout NISS file, the Transmittal file and the Summary file.

**One notebook, two run modes.** The same code path serves both modes. Only the reporting period changes.

| Parameter | Meaning | Example |
|---|---|---|
| `run_mode` | `YEARLY` or `MONTHLY` | `MONTHLY` |
| `report_year` | Four digit year. Blank = auto (YEARLY: previous year, MONTHLY: previous month) | `2026` |
| `report_month` | Month number 1 to 12, used by MONTHLY only. Blank = auto | `4` |
| `liability_reserve_as_of` | Optional `YYYY-MM`. Month used to value liability reserves. Blank = last month of the period | `2026-03` |
| `territory_code_file` | Optional override. Blank = `Files/Territory_code_<CallYear>.csv`, falling back to the previous two years | blank |
| `run_validations` | `true` enables row-count and reconciliation checks | `true` |
| `strict_preflight` | `true` stops the run if source data does not cover the period | `true` |

**How the period drives the calculations**

| Item | YEARLY (example 2025) | MONTHLY (example 2026-04) |
|---|---|---|
| Policy BookDate window | 2025-01 to 2025-12 | 2026-04 |
| Paid loss and ALAE window (BeginningOfMonth) | 2025-01-01 to 2025-12-01 | 2026-04-01 |
| "Prior period" cut-off for paid and reserve counts | before 2025-01-01 | before 2026-04-01 |
| "End of period" cut-off | before 2026-01-01 | before 2026-05-01 |
| Reserve month (property) | 2025-12-01 | 2026-04-01 |
| CalendarYear / CallYear in record | 25 / 26 | 26 / 27 |

**Output isolation.** YEARLY keeps the legacy table and folder names so existing consumers are unaffected. MONTHLY writes to period suffixed tables (for example `Final_Transformed_Data_M202604`) and period folders (for example `Files/NISS_Reports/Monthly/202604/`), so a monthly run can never overwrite a yearly output.

**Legacy behaviors preserved on purpose.** See the markdown cell "Legacy behaviors preserved" before the claim to policy stage.

## 1. Parameters

In [ ]:
# PARAMETERS CELL
# In Fabric, mark this cell with "Toggle parameter cell" so that a Data Pipeline
# Notebook activity can override these values through Base parameters.
# All values are strings on purpose, because pipelines pass parameters as text.

run_mode = "YEARLY"                    # "YEARLY" or "MONTHLY"
report_year = "2025"                   # "2025"; blank = auto
report_month = ""                      # "1" to "12" (MONTHLY only); blank = auto
liability_reserve_as_of = "2026-03"    # "YYYY-MM"; blank = last month of the period
territory_code_file = ""               # blank = Files/Territory_code_<CallYear>.csv
run_validations = "true"               # "true" or "false"
strict_preflight = "true"              # "true" or "false"

## 2. Imports, reporting period and run context

The reporting period, the liability reserve month and the output names are resolved here. Every later cell reads them from `ctx`, so no date or year is hard coded anywhere else.

In [ ]:
import calendar
import datetime as dt
import time
from dataclasses import dataclass
from functools import reduce
from typing import Dict, List, Optional, Tuple

from pyspark import StorageLevel
from pyspark.sql import Column, DataFrame, Window
from pyspark.sql import functions as F

NOTEBOOK_START_TIME = time.time()


def log(message: str) -> None:
    """Print a timestamped log line.

    Args:
        message: Text to write to the notebook output.
    """
    print(f"[{dt.datetime.now():%Y-%m-%d %H:%M:%S}] {message}")


def to_bool(value) -> bool:
    """Convert a pipeline parameter (text or bool) to a boolean.

    Args:
        value: Parameter value such as "true", "False", "1" or True.

    Returns:
        True when the value represents an enabled flag, otherwise False.
    """
    return str(value).strip().lower() in ("1", "true", "yes", "y")

In [ ]:
RUN_MODE_YEARLY = "YEARLY"
RUN_MODE_MONTHLY = "MONTHLY"

# YEARLY runs keep the legacy table and folder names so that downstream
# consumers of the yearly outputs continue to work without changes.
USE_LEGACY_NAMES_FOR_YEARLY = True


def add_months(first_of_month: dt.date, months: int) -> dt.date:
    """Add whole months to a first-of-month date.

    Args:
        first_of_month: A date whose day is 1.
        months: Number of months to add (can be negative).

    Returns:
        The first day of the resulting month.
    """
    month_index = first_of_month.year * 12 + (first_of_month.month - 1) + months
    return dt.date(month_index // 12, month_index % 12 + 1, 1)


@dataclass(frozen=True)
class ReportingPeriod:
    """Immutable description of the reporting period.

    Attributes:
        run_mode: YEARLY or MONTHLY.
        start: First day of the first month in the period.
        end_exclusive: First day after the period.
    """

    run_mode: str
    start: dt.date
    end_exclusive: dt.date

    @property
    def last_month_start(self) -> dt.date:
        """First day of the last month inside the period."""
        return add_months(self.end_exclusive, -1)

    @property
    def end_inclusive(self) -> dt.date:
        """Last calendar day inside the period."""
        return self.end_exclusive - dt.timedelta(days=1)

    @property
    def start_ym(self) -> int:
        """First month as a YYYYMM integer."""
        return self.start.year * 100 + self.start.month

    @property
    def end_ym(self) -> int:
        """Last month as a YYYYMM integer."""
        last = self.last_month_start
        return last.year * 100 + last.month

    @property
    def calendar_year(self) -> int:
        """Calendar year written in the NISS record."""
        return self.start.year

    @property
    def call_year(self) -> int:
        """Call year written in the NISS record (calendar year plus one)."""
        return self.start.year + 1

    @property
    def label(self) -> str:
        """Short label used in names: 2025 for YEARLY, 202604 for MONTHLY."""
        if self.run_mode == RUN_MODE_YEARLY:
            return str(self.start.year)
        return f"{self.start.year}{self.start.month:02d}"


def _clean(value) -> str:
    """Return a stripped string, treating None as blank."""
    return "" if value is None else str(value).strip()


def _to_int(text: str, name: str) -> int:
    """Convert parameter text to int, accepting values such as '2025.0'."""
    try:
        return int(float(text))
    except ValueError as exc:
        raise ValueError(f"Parameter {name} must be numeric, received '{text}'") from exc


def resolve_reporting_period(
    run_mode: str,
    report_year: str,
    report_month: str,
    today: Optional[dt.date] = None,
) -> ReportingPeriod:
    """Resolve run parameters into a concrete reporting period.

    Auto rules when values are blank:
        YEARLY  -> previous calendar year.
        MONTHLY -> previous calendar month.
        MONTHLY with only report_month -> most recent past occurrence of that month.

    Args:
        run_mode: YEARLY or MONTHLY (case insensitive).
        report_year: Four digit year or blank.
        report_month: Month number 1 to 12 or blank (ignored for YEARLY).
        today: Override for the current date (used for testing).

    Returns:
        A ReportingPeriod.

    Raises:
        ValueError: If a parameter is invalid or inconsistent.
    """
    mode = _clean(run_mode).upper()
    if mode not in (RUN_MODE_YEARLY, RUN_MODE_MONTHLY):
        raise ValueError(f"run_mode must be YEARLY or MONTHLY, received '{run_mode}'")

    today = today or dt.date.today()
    year_text, month_text = _clean(report_year), _clean(report_month)

    if mode == RUN_MODE_YEARLY:
        if month_text:
            log(f"WARNING: report_month '{month_text}' is ignored in YEARLY mode")
        year = _to_int(year_text, "report_year") if year_text else today.year - 1
        if not 2000 <= year <= 2100:
            raise ValueError(f"report_year {year} is outside the supported range")
        return ReportingPeriod(mode, dt.date(year, 1, 1), dt.date(year + 1, 1, 1))

    if year_text and month_text:
        year = _to_int(year_text, "report_year")
        month = _to_int(month_text, "report_month")
    elif not year_text and not month_text:
        previous = add_months(today.replace(day=1), -1)
        year, month = previous.year, previous.month
    elif month_text:
        month = _to_int(month_text, "report_month")
        year = today.year if month < today.month else today.year - 1
    else:
        raise ValueError("report_month is required when report_year is supplied in MONTHLY mode")

    if not 1 <= month <= 12:
        raise ValueError(f"report_month {month} must be between 1 and 12")
    if not 2000 <= year <= 2100:
        raise ValueError(f"report_year {year} is outside the supported range")
    start = dt.date(year, month, 1)
    return ReportingPeriod(mode, start, add_months(start, 1))


def resolve_liability_valuation_month(raw_value: str, period: ReportingPeriod) -> dt.date:
    """Resolve the month used to value liability reserves.

    Liability reserves (EAS LOB 193 and 194) can be valued later than the end of
    the period. Payments and ALAE made between the end of the period and the
    valuation month are added to the reserve amount. Blank means no lag.

    Args:
        raw_value: Text such as "2026-03", "202603", "2026-03-01" or blank.
        period: The resolved reporting period.

    Returns:
        First day of the valuation month.

    Raises:
        ValueError: If the value cannot be parsed or is earlier than the period end.
    """
    text = _clean(raw_value)
    if not text:
        return period.last_month_start
    digits = text.replace("-", "")[:6]
    if len(digits) != 6 or not digits.isdigit():
        raise ValueError(f"liability_reserve_as_of must look like YYYY-MM, received '{raw_value}'")
    valuation = dt.date(int(digits[:4]), int(digits[4:6]), 1)
    if valuation < period.last_month_start:
        raise ValueError(
            f"liability_reserve_as_of {valuation:%Y-%m} is earlier than the last month of the "
            f"period ({period.last_month_start:%Y-%m}). Clear the parameter or move it forward."
        )
    return valuation


@dataclass(frozen=True)
class RunContext:
    """Everything that depends on the run parameters.

    Attributes:
        period: Resolved reporting period.
        liability_valuation_month: First day of the liability reserve valuation month.
        use_legacy_names: Keep legacy table and folder names for YEARLY runs.
    """

    period: ReportingPeriod
    liability_valuation_month: dt.date
    use_legacy_names: bool = True

    @property
    def liability_valuation_end_exclusive(self) -> dt.date:
        """First day after the liability valuation month."""
        return add_months(self.liability_valuation_month, 1)

    @property
    def _is_legacy(self) -> bool:
        return self.period.run_mode == RUN_MODE_YEARLY and self.use_legacy_names

    def table_name(self, base_name: str) -> str:
        """Return the period specific table name for a base table name."""
        if self._is_legacy:
            return base_name
        prefix = "Y" if self.period.run_mode == RUN_MODE_YEARLY else "M"
        return f"{base_name}_{prefix}{self.period.label}"

    def output_path(self, folder_name: str) -> str:
        """Return the period specific output folder under Files/."""
        if self._is_legacy:
            return f"Files/{folder_name}"
        mode_folder = "Yearly" if self.period.run_mode == RUN_MODE_YEARLY else "Monthly"
        return f"Files/NISS_Reports/{mode_folder}/{self.period.label}/{folder_name}"


RUN_VALIDATIONS = to_bool(run_validations)
STRICT_PREFLIGHT = to_bool(strict_preflight)

_period = resolve_reporting_period(run_mode, report_year, report_month)
ctx = RunContext(
    period=_period,
    liability_valuation_month=resolve_liability_valuation_month(liability_reserve_as_of, _period),
    use_legacy_names=USE_LEGACY_NAMES_FOR_YEARLY,
)

if ctx.period.run_mode == RUN_MODE_MONTHLY and ctx.liability_valuation_month > ctx.period.last_month_start:
    log("WARNING: liability_reserve_as_of is later than the period end in a MONTHLY run. "
        "Clear it unless a lagged liability valuation is intended.")

log(f"Run mode                  : {ctx.period.run_mode}")
log(f"Reporting period          : {ctx.period.start} to {ctx.period.end_inclusive}")
log(f"Calendar year / Call year : {ctx.period.calendar_year} / {ctx.period.call_year}")
log(f"Liability reserve month   : {ctx.liability_valuation_month:%Y-%m}")
log(f"Final table example       : {ctx.table_name('Final_Transformed_Data')}")
log(f"Output folder example     : {ctx.output_path('First_NISS_FORMAT_FILE')}")

## 3. Constants, data fix registers and coverage mapping

Record specific corrections that were hard coded in the original logic are listed in registers. Add or remove entries here without touching the transformation code.

In [ ]:
# Source schema and business constants. Nothing in this cell depends on the period.

ODS_SCHEMA = "ODS_STATS_TBLS_LH"

CLAIMS_COMPANY_CODE = "14044"
POLICY_LOB_CODE = "AC"
LIABILITY_EAS_LOB_CODES = ("193", "194")
PROPERTY_EAS_LOB_CODE = "212"
ALLOCATED_LOSS_EAS_LOB_CODES = ("193", "194", "212")

# Policies whose term started before this date are out of scope for the STAT call.
TERM_EFFECTIVE_CUTOFF = dt.date(2023, 7, 1)

# Vehicle class codes excluded from risk level coverages.
EXCLUDED_VEHICLE_CLASS_CODES = ("9410", "9437", "9582", "9235", "9670")

# Primary risk state mapping to the NISS state code.
ALLOWED_TERRITORY_STATES = ["DE", "IL", "IN", "KS", "OH", "OK", "PA", "VA"]
NISS_STATE_CODES = {
    "DE": "07", "IL": "12", "IN": "13", "KS": "15",
    "OH": "34", "OK": "35", "PA": "37", "VA": "45",
}

# Constant values for the NISS record. Seeded into the Constant table only when
# a key is missing, so manual edits made to the table are respected.
NISS_CONSTANT_DEFAULTS = {
    "CompanyNumber": "526",
    "LineCode": "1",
    "CommercialIndicatorCode": "9",
    "ClaimantLevelIndicator": "C",
}
CONSTANT_TABLE = "Constant"

# Coverage pairs merged into the primary coverage (secondary amounts are added
# to the primary row and the secondary row is removed).
COVERAGE_MERGE_PAIRS = [
    ("BODINJLIAB", "BI"),
    ("COLLISION", "COLL"),
    ("OTC", "COMP"),
]

# Lookup files stored under the Files section of the default lakehouse.
LOOKUP_FILES = {
    "duplicate_coverages": "Files/DeleteDuplicateCoveragesFromSTATExtract.csv",
    "aslob_manual_update": "Files/ASOLB_Manual_update.csv",
    "ba_plus_aslob": "Files/BA PLUS-UpdateASLOB.csv",
    "missing_zip": "Files/Missing_ZIP_Code.csv",
    "zip_further_fix": "Files/ZIP_Further_fix.csv",
    "has_pip_vuwc": "Files/HasPIPVUWC_New.csv",
    "niss_column_mapping": "Files/NISS_Column_Mapping-Updated.csv",
    "corrected_class_code": "Files/Corrected_Class_code.csv",
    "niss_cause_of_loss": "Files/NissCauseofloss_df.csv",
    "exception_code_mapping": "Files/Exception_code_Mapping.csv",
    "datafix_offset": "Files/DataFix-ForOffsetRecordsWithoutReferenceData.csv",
    "datafix_claims": "Files/DataFix-Claims.csv",
    "classes_with_claim_ind": "Files/ClassesWithClaimInd_9.csv",
}

# Legacy behavior switch. The original yearly notebook removed every claim row
# whose ReserveAmount is NULL or negative. Paid loss and ALAE rows carry a NULL
# ReserveAmount, so they were removed as well. False keeps that behavior and
# therefore keeps the yearly results unchanged. True keeps paid and ALAE rows
# and removes only negative reserves. Change only after business confirmation.
KEEP_NON_RESERVE_CLAIM_ROWS = False

In [ ]:
# Manual data fix registers.
# Every record specific correction that used to be hard coded inside the logic
# lives here. Add or remove entries here without touching the transformation code.

# Policy versions removed from the base extract: (PolicyNumber, PolicyVersion).
MANUAL_POLICY_VERSION_EXCLUSIONS = [
    ("GMCA100000107", 1449),
    ("GMCA100000080", 4740),
]

# Offset reference rows matching a rule are kept only when they are child
# coverages (ParentCoverageId is not null).
OFFSET_REFERENCE_CHILD_ONLY_RULES = [
    {"PolicyNumber": "GMCA001142870", "CoverageCode": "BROAD PIP", "ClaimCode": "BROAD PIP__NF"},
]

# Overrides for DerivedClaimCoverageName on unmatched claim rows.
DERIVED_COVERAGE_NAME_OVERRIDES = [
    {"match": {"PolicyNumber": 134028, "TransactionNumber": 26,
               "ClaimNumber": 596598, "ClaimCoverageName": "MED EXP"},
     "value": "PIP LIMIT"},
]

# Overrides for ZipCode by InTransactionId.
ZIP_CODE_OVERRIDES = [
    {"match": {"InTransactionId": 2830107}, "value": 45356},
]

# Overrides for CoverageCode on specific claims.
CLAIM_COVERAGE_CODE_OVERRIDES = [
    {"match": {"ClaimNumber": 633662, "ClaimCoverageName": "BA PLUS_RR"}, "value": "BA PLUS"},
]

# Overrides for NISS_Exception_Code on specific coverage rows.
EXCEPTION_CODE_OVERRIDES = [
    {"match": {"InTransactionId": 3203888, "RiskItemId": 2174291,
               "PolicyNumber": "GMCA100000183", "PolicyVersion": 1824,
               "CoverageId": 17000057, "ClaimCode": "INC LOSS__NF"},
     "value": "53"},
    {"match": {"InTransactionId": 3276779, "RiskItemId": 2216881,
               "PolicyNumber": "GMCA001138996", "PolicyVersion": 2215,
               "CoverageId": 17390445, "ClaimCode": "INC LOSS__NF"},
     "value": "53"},
    {"match": {"InTransactionId": 156722, "RiskItemId": 92935,
               "PolicyNumber": "135096", "PolicyVersion": 12,
               "CoverageId": 653539, "ClaimCode": "INC LOSS__NF"},
     "value": "52"},
]

In [ ]:
# Claim coverage mapping: (ClaimCoverageName, CoverageExtension) -> DerivedClaimCoverageName.
# This single definition replaces the CoverageMapping CTE that was copied into
# four SQL statements in the original notebook.
COVERAGE_MAPPING_ROWS = [
    ('ACC DEATH', 'NF', 'ACC DEATH_NF'),
    ('BA PLUS', 'CM', 'BA PLUS_TE'),
    ('BA PLUS', 'RR', 'BA PLUS_RR'),
    ('BA PLUS', 'TL', 'BA PLUS_T-L'),
    ('BA PLUS_RR', 'CM', 'BA PLUS_RR'),
    ('BA PLUS_T-L', 'CM', 'BA PLUS_T-L'),
    ('BI', 'BI', 'BODINJLIAB'),
    ('BI', 'CBI', 'BODINJLIAB'),
    ('BODINJLIAB', 'BI', 'BODINJLIAB'),
    ('BODINJLIAB', 'CBI', 'BODINJLIAB'),
    ('COLL', 'CL', 'COLLISION'),
    ('COLLISION', 'CL', 'COLLISION'),
    ('COMB-FPB', 'FPB', 'COMB-FPB'),
    ('COMP', 'CM', 'OTC'),
    ('FRMTRKPOLL', 'PD', 'FRMTRKPOLL_PRPDMGLIAB'),
    ('GK COLL', 'CL', 'GK COLL'),
    ('GK COMP', 'CM', 'GK COMP'),
    ('PIP LIMIT_INC LOSS', 'NF', 'PIP LIMIT_INC LOSS'),
    ('PIP LIMIT_ACC DEATH', 'NF', 'PIP LIMIT_ACC DEATH'),
    ('PIP LIMIT_FUN EXP', 'NF', 'PIP LIMIT_FUN EXP'),
    ('HIRED AUTO', 'PD', 'HIRED AUTO_PRPDMGLIAB'),
    ('HIRED PHYS', 'CM', 'HIRED PHYS_OTC'),
    ('INC LOSS', 'NF', 'INC LOSS'),
    ('MED EXP', 'FPB', 'MED EXP'),
    ('MED EXP', 'NF', 'MED EXP'),
    ('MEDPAYMENT', 'MP', 'MEDPAYMENT'),
    ('MP', 'MP', 'MEDPAYMENT'),
    ('OTC', 'CM', 'OTC'),
    ('PD', 'CPD', 'PRPDMGLIAB'),
    ('PD', 'PD', 'PRPDMGLIAB'),
    ('PIP LIMIT', 'NF', 'PIP LIMIT'),
    ('PIP LIMIT_MED EXP', 'NF', 'PIP LIMIT_MED EXP'),
    ('PRPDMGLIAB', 'CPD', 'PRPDMGLIAB'),
    ('PRPDMGLIAB', 'PD', 'PRPDMGLIAB'),
    ('RR', 'CM', 'RR'),
    ('RR', 'RR', 'RR'),
    ('T&L', 'TL', 'T-L'),
    ('T-L', 'CM', 'T-L'),
    ('UBI', 'UBI', 'UBI'),
    ('UBI NS', 'UBI', 'UBI'),
    ('UIM NS', 'UBI', 'UIM'),
    ('UIM ST', 'UBI', 'UBI'),
    ('UPD', 'UPD', 'UPD'),
]

# The post period payment correction query used two different derived names in
# the original notebook. They are kept exactly as they were.
CORRECTION_MAPPING_OVERRIDES = {
    ("RR", "RR"): "BA PLUS_RR",
    ("T&L", "TL"): "BA PLUS_T-L",
}


def register_coverage_mapping_views() -> None:
    """Register the CoverageMapping and CoverageMappingCorrection temp views."""
    columns = ["ClaimCoverageName", "CoverageExtension", "DerivedClaimCoverageName"]
    spark.createDataFrame(COVERAGE_MAPPING_ROWS, columns).createOrReplaceTempView("CoverageMapping")

    correction_rows = [
        (name, extension, CORRECTION_MAPPING_OVERRIDES.get((name, extension), derived))
        for name, extension, derived in COVERAGE_MAPPING_ROWS
    ]
    spark.createDataFrame(correction_rows, columns).createOrReplaceTempView("CoverageMappingCorrection")


register_coverage_mapping_views()

## 4. Helpers, pre-flight checks and SQL builders

The SQL statements are generated from the reporting period. For the 2025 yearly run they render to the same SQL as the original notebook (apart from the intended generalizations).

In [ ]:
def read_lookup_csv(path: str) -> DataFrame:
    """Read a lookup CSV from the lakehouse Files section.

    Args:
        path: Path such as "Files/Missing_ZIP_Code.csv".

    Returns:
        DataFrame with header and inferred schema.
    """
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(path)
    )


def write_delta_table(df: DataFrame, table_name: str) -> None:
    """Overwrite a managed Delta table, allowing schema changes.

    Args:
        df: Data to persist.
        table_name: Target table in the default lakehouse.
    """
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )


def write_csv_folder(df: DataFrame, path: str, header: bool = False) -> None:
    """Write a single file CSV folder (no quoting changes, overwrite mode).

    Args:
        df: Data to write.
        path: Target folder under Files/.
        header: Whether to write a header row.
    """
    (
        df.coalesce(1)
        .write.mode("overwrite")
        .option("header", "true" if header else "false")
        .csv(path)
    )


def build_match_condition(match: Dict[str, object]) -> Column:
    """Build an AND condition from a column to value dictionary.

    Args:
        match: Mapping of column name to the value it must equal.

    Returns:
        A Spark Column expression.
    """
    return reduce(
        lambda left, right: left & right,
        [F.col(column) == value for column, value in match.items()],
    )


def apply_override_rules(df: DataFrame, column: str, rules: List[Dict]) -> DataFrame:
    """Overwrite a column where a rule matches and keep the value otherwise.

    Args:
        df: Input data.
        column: Column to overwrite.
        rules: List of {"match": {...}, "value": ...} dictionaries.

    Returns:
        Data with the column updated. Unchanged when rules is empty.
    """
    if not rules:
        return df
    expression = None
    for rule in rules:
        condition = build_match_condition(rule["match"])
        value = F.lit(rule["value"])
        expression = F.when(condition, value) if expression is None else expression.when(condition, value)
    return df.withColumn(column, expression.otherwise(F.col(column)))


def as_date(value) -> Optional[dt.date]:
    """Convert a date, datetime or ISO text value to a date.

    Args:
        value: Value returned by a Spark aggregation.

    Returns:
        A date, or None when the value is empty.
    """
    if value is None:
        return None
    if isinstance(value, dt.datetime):
        return value.date()
    if isinstance(value, dt.date):
        return value
    return dt.date.fromisoformat(str(value)[:10])


def path_exists(path: str) -> Optional[bool]:
    """Check whether a lakehouse path exists.

    Args:
        path: Path such as "Files/Territory_code_2026.csv".

    Returns:
        True or False, or None when the check itself is not available.
    """
    try:
        return bool(notebookutils.fs.exists(path))  # noqa: F821 (provided by Fabric)
    except Exception:
        return None

In [ ]:
def resolve_territory_file(context: RunContext, override: str) -> str:
    """Return the territory lookup file for the run.

    Order of preference:
        1. The territory_code_file parameter when supplied.
        2. Files/Territory_code_<CallYear>.csv.
        3. The same file name for the previous two years, so a monthly run early
           in a year can still use the latest published territory file.

    Args:
        context: Run context.
        override: Value of the territory_code_file parameter (blank = auto).

    Returns:
        Path of the territory lookup file.

    Raises:
        FileNotFoundError: If no candidate file is confirmed to exist.
    """
    if override.strip():
        return override.strip()

    candidates = [f"Files/Territory_code_{context.period.call_year - offset}.csv" for offset in range(3)]
    for path in candidates:
        exists = path_exists(path)
        if exists is None:
            return candidates[0]  # existence check not available, trust the primary name
        if exists:
            if path != candidates[0]:
                log(f"WARNING: {candidates[0]} not found, using {path}")
            return path
    raise FileNotFoundError(
        f"No territory lookup file found (tried {', '.join(candidates)}). "
        f"Upload one or set the territory_code_file parameter."
    )


def run_preflight_checks(context: RunContext, strict: bool) -> None:
    """Stop early when source data does not cover the requested period.

    Claims must be loaded up to the later of the period end and the liability
    valuation month. Otherwise reserves and payments would silently be zero.

    Args:
        context: Run context.
        strict: Raise on failure when True, otherwise log a warning.

    Raises:
        RuntimeError: When strict and the claims data is not loaded far enough.
    """
    required_month = max(context.period.last_month_start, context.liability_valuation_month)
    latest_claim_month = as_date(
        spark.sql(
            f"SELECT MAX(BeginningOfMonth) AS m FROM DMAnalytics.FactClaimsMonthlyDataStore "
            f"WHERE CompanyCode = '{CLAIMS_COMPANY_CODE}'"
        ).first()["m"]
    )
    log(f"Latest claim month loaded : {latest_claim_month}  (required: {required_month})")
    if latest_claim_month is None or latest_claim_month < required_month:
        message = (f"FactClaimsMonthlyDataStore is loaded only up to {latest_claim_month}, "
                   f"but {required_month} is required for this run.")
        if strict:
            raise RuntimeError(message)
        log(f"WARNING: {message}")

    latest_book_date = as_date(
        spark.sql(f"SELECT MAX(BookDate) AS m FROM {ODS_SCHEMA}.policy_policy_detail_info").first()["m"]
    )
    log(f"Latest policy BookDate    : {latest_book_date}")
    if latest_book_date is not None and latest_book_date < context.period.end_inclusive:
        log("WARNING: policy data does not yet reach the end of the period. "
            "The run will contain a partial period.")


TERRITORY_FILE = resolve_territory_file(ctx, territory_code_file)
run_preflight_checks(ctx, STRICT_PREFLIGHT)

In [ ]:
def _book_date_filter(period: ReportingPeriod) -> str:
    """SQL predicate selecting policy rows booked inside the period."""
    return (f"(YEAR(PDI.BookDate) * 100 + MONTH(PDI.BookDate) "
            f"BETWEEN {period.start_ym} AND {period.end_ym})")


def build_policy_extract_sql(period: ReportingPeriod) -> str:
    """Build the base policy extract (onset and offset coverage rows).

    The four coverage levels (risk, location, policy, premium on child) share
    the same output columns and the same filters. They differ only in how the
    coverage row is attached.

    Args:
        period: Reporting period used for the BookDate filter.

    Returns:
        SQL text for spark.sql.
    """
    book_date = _book_date_filter(period)
    term_cutoff = (TERM_EFFECTIVE_CUTOFF - dt.timedelta(days=1)).isoformat()
    excluded_classes = ",".join(f"'{code}'" for code in EXCLUDED_VEHICLE_CLASS_CODES)

    transaction_columns = """
        PDI.PolicyNumber, PDI.PolicyVersion, PDI.InTransactionId, PDI.ReferencePolicyVersion,
        PDI.PolicySource, PDI.BookDate, PDI.ProrataPercent, PI.LOBCode,"""
    risk_columns = "R.RiskItemType, R.UnitNumber, R.ClassCode,"
    null_risk_columns = """CAST(NULL AS VARCHAR(50)) AS RiskItemType,
        CAST(NULL AS INT) AS UnitNumber, CAST(NULL AS VARCHAR(20)) AS ClassCode,"""
    coverage_columns = """
        PDI.PrimaryRiskState,
        CI.CoverageId, CI.ParentCoverageId, CI.RiskItemId, CI.ProductId, CI.CoverageCode,
        CI.CoverageExtension, CI.CoverageGroup, CI.OnsetAmount, CI.OffsetAmount, CI.Limit1,
        CI.Limit2, CI.ASLOB, CI.Option1, CI.Exposure, CI.Deductible1, CI.ClaimCode,
        CI.CoverageInfoGUID"""
    product_join = f"""
    FROM {ODS_SCHEMA}.policy_policy_detail_info PDI
    JOIN {ODS_SCHEMA}.policy_Product_info PI
        ON PI.InTransactionId = PDI.InTransactionId
       AND PI.LOBCode = '{POLICY_LOB_CODE}'
       AND PI.ParentProductID IS NULL"""
    where_clause = f"""
    WHERE {book_date}
      AND ((CI.OnsetAmount IS NOT NULL AND CI.OnsetAmount <> 0)
        OR (CI.OffsetAmount IS NOT NULL AND CI.OffsetAmount <> 0))
      AND PDI.TermEffectiveDate > '{term_cutoff}'"""

    def risk_join(risk_type: str, class_filter: str) -> str:
        return f"""
    JOIN {ODS_SCHEMA}.policy_risk_item_info R
        ON R.InTransactionId = PDI.InTransactionId
       AND R.RiskItemType = '{risk_type}'{class_filter}
    JOIN {ODS_SCHEMA}.policy_coverage_info CI
        ON CI.InTransactionId = PDI.InTransactionId
       AND CI.RiskItemId = R.RiskItemId
       AND CI.ProductId IS NULL"""

    vehicle_filter = f"\n       AND R.ClassCode NOT IN ({excluded_classes})"
    policy_level_join = f"""
    JOIN {ODS_SCHEMA}.policy_coverage_info CI
        ON CI.InTransactionId = PDI.InTransactionId
       AND CI.ProductId = PI.ProductId
       AND CI.RiskItemId IS NULL"""
    premium_on_child_join = f"""
    JOIN {ODS_SCHEMA}.policy_coverage_info CI
        ON CI.InTransactionId = PDI.InTransactionId
       AND CI.ProductId IS NULL
       AND CI.RiskItemId IS NULL"""

    def cte(name: str, risk_select: str, joins: str) -> str:
        return (f"{name} AS (SELECT{transaction_columns}\n        {risk_select}"
                f"{coverage_columns}{product_join}{joins}{where_clause})")

    ctes = ",\n".join([
        cte("Risk_Level_Coverages", risk_columns, risk_join("Vehicle", vehicle_filter)),
        cte("Location_Level_Coverages", risk_columns, risk_join("Location", "")),
        cte("Policy_Level_Coverages", null_risk_columns, policy_level_join),
        cte("Prem_On_Child_Coverages", null_risk_columns, premium_on_child_join),
    ])
    return f"""
WITH {ctes}
SELECT DISTINCT *
FROM (
    SELECT * FROM Risk_Level_Coverages
    UNION ALL SELECT * FROM Location_Level_Coverages
    UNION ALL SELECT * FROM Policy_Level_Coverages
    UNION ALL SELECT * FROM Prem_On_Child_Coverages
) U
"""


def build_claims_sql(
    *,
    payment_upper: dt.date,
    incurred_upper: dt.date,
    prior_cutoff: dt.date,
    payment_end_cutoff: dt.date,
    incurred_end_cutoff: dt.date,
    reserve_month: dt.date,
    start: dt.date,
    lob_filter: str,
    accident_filter: str,
) -> str:
    """Build the claim measure query shared by liability and property claims.

    All date arguments are compared with FactClaimsMonthlyDataStore.BeginningOfMonth.

    Args:
        payment_upper: Last BeginningOfMonth value of the payment window.
        incurred_upper: Upper bound of the incurred window (informational column).
        prior_cutoff: Amounts before this date are "prior period".
        payment_end_cutoff: Payments before this date are "as of end of period".
        incurred_end_cutoff: Incurred before this date is "as of end of period".
        reserve_month: BeginningOfMonth value used for the reserve amount.
        start: First BeginningOfMonth value of the period.
        lob_filter: SQL predicate on EASLobCode.
        accident_filter: Optional SQL predicate on LossDate (starts with AND) or blank.

    Returns:
        SQL text for spark.sql.
    """
    return f"""
SELECT
    a.PolicyNumber,
    a.TransactionNumber,
    a.ClaimNumber,
    a.ClaimantNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', '') as ItemNumber,
    a.ClaimCoverageName,
    a.CoverageExtension,
    LossDate,
    Year(a.LossDate) as AccidentYear,
    m.DerivedClaimCoverageName,
    Cast(0 as decimal(2)) as PaidCount,
    Cast(0 as decimal(2)) as ReserveCount,
    SUM(CASE WHEN BeginningOfMonth BETWEEN '{start}' AND '{payment_upper}'
             THEN PaymentAmount ELSE 0 END) AS PaymentAmountDuration,
    SUM(CASE WHEN BeginningOfMonth < '{prior_cutoff}'
             THEN PaymentAmount ELSE 0 END) AS PaymentAmountDurationAsOfPriorPeriod,
    SUM(CASE WHEN BeginningOfMonth < '{payment_end_cutoff}'
             THEN PaymentAmount ELSE 0 END) AS PaymentAmountDurationAsOfEndOfPeriod,
    SUM(CASE WHEN BeginningOfMonth BETWEEN '{start}' AND '{incurred_upper}'
             THEN LossIncurredNoIBNR ELSE 0 END) AS LossIncurredNOIBNRDuration,
    SUM(CASE WHEN BeginningOfMonth < '{prior_cutoff}'
             THEN LossIncurredNoIBNR ELSE 0 END) AS LossIncurredNOIBNRAsOfPriorPeriod,
    SUM(CASE WHEN BeginningOfMonth < '{incurred_end_cutoff}'
             THEN LossIncurredNoIBNR ELSE 0 END) AS LossIncurredNOIBNRAsOfEndOfPeriod,
    SUM(CASE WHEN BeginningOfMonth = '{reserve_month}'
             THEN NetReserveAmount + ExcessDirectAmount ELSE 0 END) AS FinalReserveAmount
FROM DMAnalytics.FactClaimsMonthlyDataStore a
LEFT JOIN CoverageMapping m
    ON a.ClaimCoverageName = m.ClaimCoverageName
   AND a.CoverageExtension = m.CoverageExtension
WHERE CompanyCode = '{CLAIMS_COMPANY_CODE}'
  AND {lob_filter}{accident_filter}
GROUP BY
    a.PolicyNumber, a.TransactionNumber, a.ClaimNumber, a.ClaimantNumber, a.ItemNumber,
    a.ClaimCoverageName, a.CoverageExtension, m.DerivedClaimCoverageName,
    a.LossDate, a.PropertyOrLiability
"""


def build_liability_claims_sql(context: RunContext) -> str:
    """Claim measures for liability lines (EAS LOB 193 and 194).

    Liability reserves are valued at the liability valuation month. Only
    accidents on or before the end of the period are included.
    """
    period = context.period
    lob_codes = ", ".join(f"'{code}'" for code in LIABILITY_EAS_LOB_CODES)
    return build_claims_sql(
        payment_upper=period.last_month_start,
        incurred_upper=period.end_inclusive,
        prior_cutoff=period.start,
        payment_end_cutoff=period.end_exclusive,
        incurred_end_cutoff=context.liability_valuation_end_exclusive,
        reserve_month=context.liability_valuation_month,
        start=period.start,
        lob_filter=f"EASLobCode in ({lob_codes})",
        accident_filter=f"\n  and (Year(LossDate) * 100 + Month(LossDate)) <= {period.end_ym}",
    )


def build_property_claims_sql(context: RunContext) -> str:
    """Claim measures for property lines (EAS LOB 212), valued at the period end."""
    period = context.period
    return build_claims_sql(
        payment_upper=period.end_inclusive,
        incurred_upper=period.end_inclusive,
        prior_cutoff=period.start,
        payment_end_cutoff=period.end_exclusive,
        incurred_end_cutoff=period.end_exclusive,
        reserve_month=period.last_month_start,
        start=period.start,
        lob_filter=f"EASLobCode = '{PROPERTY_EAS_LOB_CODE}'",
        accident_filter="",
    )


def build_post_period_payment_sql(context: RunContext) -> str:
    """Payments plus ALAE made after the period end up to the liability valuation month.

    The result is added to the liability reserve amount. When the valuation
    month equals the last month of the period the window is empty.
    """
    period = context.period
    lob_codes = ", ".join(f"'{code}'" for code in LIABILITY_EAS_LOB_CODES)
    return f"""
SELECT
    a.EASLobCode,
    a.PolicyNumber,
    a.TransactionNumber,
    a.ClaimNumber,
    a.ClaimantNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', '') as ItemNumber,
    a.ClaimCoverageName,
    a.CoverageExtension,
    Year(a.LossDate) as AccidentYear,
    m.DerivedClaimCoverageName,
    sum(a.PaymentAmount + a.AllocatedLossAdjExp) as PaidAmountALAEPostPeriod
FROM FactClaimsMonthlyDataStore a
LEFT JOIN CoverageMappingCorrection m
    ON a.ClaimCoverageName = m.ClaimCoverageName
   AND a.CoverageExtension = m.CoverageExtension
WHERE CompanyCode = '{CLAIMS_COMPANY_CODE}'
  AND EASLobCode in ({lob_codes})
  and (Year(LossDate) * 100 + Month(LossDate)) <= {period.end_ym}
  and BeginningOfMonth BETWEEN '{period.end_exclusive}' AND '{context.liability_valuation_month}'
GROUP BY
    a.EASLobCode, a.PolicyNumber, a.TransactionNumber, a.ClaimNumber, a.ClaimantNumber,
    a.ItemNumber, a.ClaimCoverageName, a.CoverageExtension, m.DerivedClaimCoverageName,
    a.LossDate, a.PropertyOrLiability
Having sum(a.PaymentAmount + a.AllocatedLossAdjExp) != 0
"""


def build_allocated_loss_sql(period: ReportingPeriod) -> str:
    """ALAE (allocated loss adjustment expense) paid inside the period."""
    lob_codes = ",".join(f"'{code}'" for code in ALLOCATED_LOSS_EAS_LOB_CODES)
    return f"""
SELECT
    a.PolicyNumber,
    a.TransactionNumber,
    a.ClaimNumber,
    a.ClaimantNumber,
    a.LossDate,
    Year(to_date(a.LossDate,'yyyyMMdd')) as AccidentYear,
    a.PropertyOrLiability,
    a.ClaimCoverageName,
    a.CoverageExtension,
    a.LocationSequenceNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', '') as ItemNumber,
    CAST(NULL AS DECIMAL(18,2)) as PaidLoss,
    sum(a.AllocatedLossAdjExp) as AllocatedLossAdjExp,
    CAST(NULL AS DECIMAL(18,2)) as ReserveAmount,
    m.DerivedClaimCoverageName
FROM FactClaimsMonthlyDataStore a
LEFT JOIN CoverageMapping m
    ON a.ClaimCoverageName = m.ClaimCoverageName
   AND a.CoverageExtension = m.CoverageExtension
WHERE EASLobCode IN ({lob_codes})
  AND a.BeginningOfMonth BETWEEN '{period.start}' AND '{period.last_month_start}'
  AND (a.AllocatedLossAdjExp <> 0)
GROUP BY
    a.PolicyNumber, a.TransactionNumber, a.ClaimNumber, a.ClaimantNumber,
    a.ClaimCoverageName, a.CoverageExtension, a.LocationSequenceNumber,
    REPLACE(TRIM(a.ItemNumber), '.0', ''), m.DerivedClaimCoverageName,
    a.LossDate, a.PropertyOrLiability
"""

## 5. Policy base extract

Onset and offset coverage rows booked in the period, manual exclusions, duplicate removal, secondary coverage merge and Origami prorata.

In [ ]:
def extract_policy_base(context: RunContext) -> DataFrame:
    """Extract onset and offset coverage rows booked in the period.

    Steps:
        1. Run the period specific policy extract.
        2. Remove manually excluded policy versions.
        3. Remove duplicate coverages listed in the duplicate coverage lookup.

    Args:
        context: Run context.

    Returns:
        Base policy coverage data.
    """
    data = spark.sql(build_policy_extract_sql(context.period))

    exclusions = spark.createDataFrame(
        MANUAL_POLICY_VERSION_EXCLUSIONS, "PolicyNumber STRING, PolicyVersion BIGINT"
    )
    data = data.join(exclusions, on=["PolicyNumber", "PolicyVersion"], how="left_anti")

    duplicate_coverages = read_lookup_csv(LOOKUP_FILES["duplicate_coverages"])
    key_columns = ["PolicyNumber", "PolicyVersion", "InTransactionId", "RiskItemId", "CoverageId"]
    return data.join(duplicate_coverages, on=key_columns, how="left_anti")


def merge_secondary_coverages(df: DataFrame) -> DataFrame:
    """Add secondary coverage amounts to the primary coverage.

    For each (primary, secondary) pair, when a risk item in a transaction has
    both coverages, the secondary onset and offset amounts are added to the
    primary row and the secondary row is removed. Rows are untouched when only
    one of the two coverages exists.

    Args:
        df: Base policy coverage data.

    Returns:
        Data with secondary coverages merged.
    """
    result = df
    for primary, secondary in COVERAGE_MERGE_PAIRS:
        both = (
            result.groupBy("RiskItemId", "InTransactionId")
            .agg(
                F.sum(F.when(F.col("CoverageCode") == primary, 1).otherwise(0)).alias("p_cnt"),
                F.sum(F.when(F.col("CoverageCode") == secondary, 1).otherwise(0)).alias("s_cnt"),
                F.sum(F.when(F.col("CoverageCode") == secondary, F.col("OnsetAmount")).otherwise(0)).alias("sec_onset_sum"),
                F.sum(F.when(F.col("CoverageCode") == secondary, F.col("OffsetAmount")).otherwise(0)).alias("sec_offset_sum"),
            )
            .filter((F.col("p_cnt") > 0) & (F.col("s_cnt") > 0))
            .select("RiskItemId", "InTransactionId", "sec_onset_sum", "sec_offset_sum")
        )

        joined = result.join(both, ["RiskItemId", "InTransactionId"], "left")

        updated_primary = (
            joined.filter(F.col("CoverageCode") == primary)
            .withColumn(
                "OnsetAmount",
                F.when(F.col("sec_onset_sum").isNotNull(), F.col("OnsetAmount") + F.col("sec_onset_sum"))
                .otherwise(F.col("OnsetAmount")),
            )
            .withColumn(
                "OffsetAmount",
                F.when(F.col("sec_offset_sum").isNotNull(), F.col("OffsetAmount") + F.col("sec_offset_sum"))
                .otherwise(F.col("OffsetAmount")),
            )
            .drop("sec_onset_sum", "sec_offset_sum")
        )

        remaining = (
            joined.filter(~((F.col("CoverageCode") == secondary) & F.col("sec_onset_sum").isNotNull()))
            .drop("sec_onset_sum", "sec_offset_sum")
        )

        result = remaining.filter(F.col("CoverageCode") != primary).unionByName(updated_primary)
    return result


def apply_origami_prorata(df: DataFrame) -> DataFrame:
    """Replace ProrataPercent with the question and answer value for Origami policies.

    Args:
        df: Policy coverage data after coverage merging.

    Returns:
        Data where Origami rows use the ProrataPercentage answer.
    """
    qa_lookup = (
        spark.table(f"{ODS_SCHEMA}.policy_question_answer")
        .filter(F.col("QuestionCode") == "ProrataPercentage")
        .select(
            F.col("InTransactionId").alias("QA_InTransactionId"),
            F.col("TextValue").alias("Prorata_from_QA"),
        )
        .dropDuplicates(["QA_InTransactionId"])  # prevents row explosion
    )

    joined = df.join(qa_lookup, df["InTransactionId"] == qa_lookup["QA_InTransactionId"], "left")
    updated = joined.withColumn(
        "ProrataPercent",
        F.when(F.col("PolicySource") == "Origami", F.col("Prorata_from_QA")).otherwise(F.col("ProrataPercent")),
    )
    return updated.drop("QA_InTransactionId", "Prorata_from_QA")


policy_base_df = extract_policy_base(ctx)
final_merged_df = apply_origami_prorata(merge_secondary_coverages(policy_base_df))
log("Policy base extraction defined (lazy). Next: offset reference records.")

## 6. Offset reference records

Offset rows take their coverage attributes from the transaction they reverse. The onset rows and the enriched offset rows are saved to `OnSet_And_Refrence_Offset`.

In [ ]:
# Offset reference SQL. Each statement reads the temp views created by the
# previous steps, so the names and the order matter. The SQL text is unchanged
# from the original notebook except that the schema name comes from ODS_SCHEMA.
OFFSET_SQL = {}

OFFSET_SQL["REF_TRANSACTION"] = f"""
SELECT
    o.*,
    pdi.InTransactionId AS Ref_InTransactionId,
    pdi.ReferencePolicyVersion AS Ref_ReferencePolicyVersion
FROM df_offset o
LEFT JOIN {ODS_SCHEMA}.policy_policy_detail_info pdi
    ON pdi.PolicyNumber = o.PolicyNumber
   AND pdi.PolicyVersion = o.ReferencePolicyVersion
"""

OFFSET_SQL["NORMAL_ROWS"] = f"""
SELECT *
FROM ref_tx_df
WHERE UnitNumber IS NOT NULL
  AND RiskItemId IS NOT NULL
"""

OFFSET_SQL["SPECIAL_ROWS"] = f"""
SELECT *
FROM ref_tx_df
WHERE UnitNumber IS NULL
   OR RiskItemId IS NULL
"""

OFFSET_SQL["RISK_ITEM_FIRST_BY_UNIT"] = f"""
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY InTransactionId, UnitNumber, RiskItemType
               ORDER BY RiskItemId
           ) rn
    FROM {ODS_SCHEMA}.policy_risk_item_info
) WHERE rn = 1
"""

OFFSET_SQL["NORMAL_WITH_REFERENCE_RISK"] = f"""
SELECT
    n.*,
    r.RiskItemId   AS Ref_RiskItemId,
    r.ProductId    AS Ref_ProductId,
    r.ClassCode    AS Ref_ClassCode,
    r.RiskItemType AS Ref_RiskItemType
FROM normal_df n
LEFT JOIN riskitem_filtered r
  ON r.InTransactionId = n.Ref_InTransactionId
 AND r.UnitNumber = n.UnitNumber
 AND r.RiskItemType = n.RiskItemType
"""

OFFSET_SQL["COVERAGE_FIRST_BY_RISK"] = f"""
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY InTransactionId, RiskItemId, CoverageCode
               ORDER BY ParentCoverageId, CoverageId
           ) rn
    FROM {ODS_SCHEMA}.policy_coverage_info
) WHERE rn = 1
"""

OFFSET_SQL["NORMAL_WITH_REFERENCE_COVERAGE"] = f"""
SELECT
    n.*,
    c.CoverageId AS Ref_CoverageId,
    c.ParentCoverageId AS Ref_ParentCoverageId,
    c.CoverageExtension AS Ref_CoverageExtension,
    c.CoverageGroup AS Ref_CoverageGroup,
    c.Limit1 AS Ref_Limit1,
    c.Limit2 AS Ref_Limit2,
    c.ASLOB AS Ref_ASLOB,
    c.Option1 AS Option1_ref,
    c.Exposure AS Ref_Exposure,
    c.Deductible1 AS Ref_Deductible1,
    c.ClaimCode AS Ref_ClaimCode,
    c.CoverageInfoGUID AS Ref_CoverageInfoGUID
FROM normal_risk_df n
LEFT JOIN coverage_filtered_risk c
  ON c.InTransactionId = n.Ref_InTransactionId
 AND c.RiskItemId = n.Ref_RiskItemId
 AND c.CoverageCode = n.CoverageCode
"""

OFFSET_SQL["COVERAGE_FIRST_BY_PRODUCT"] = f"""
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY InTransactionId, ProductId, CoverageCode, CoverageExtension, ClaimCode
               ORDER BY ParentCoverageId, CoverageId
           ) rn
    FROM {ODS_SCHEMA}.policy_coverage_info
) WHERE rn = 1
"""

OFFSET_SQL["SPECIAL_NON_ORIGAMI"] = f"""
SELECT * FROM special_df WHERE PolicySource <> 'Origami'
"""

OFFSET_SQL["SPECIAL_ORIGAMI"] = f"""
SELECT * FROM special_df WHERE PolicySource = 'Origami'
"""

OFFSET_SQL["SPECIAL_NON_ORIGAMI_JOINED"] = f"""
SELECT
    s.*,
    NULL AS Ref_RiskItemId,
    NULL AS Ref_ClassCode,
    NULL AS Ref_RiskItemType,
    s.ProductId AS Ref_ProductId,
    c.CoverageId AS Ref_CoverageId,
    c.ParentCoverageId AS Ref_ParentCoverageId,
    c.CoverageExtension AS Ref_CoverageExtension,
    c.CoverageGroup AS Ref_CoverageGroup,
    c.Limit1 AS Ref_Limit1,
    c.Limit2 AS Ref_Limit2,
    c.ASLOB AS Ref_ASLOB,
    c.Option1 AS Option1_ref,
    c.Exposure AS Ref_Exposure,
    c.Deductible1 AS Ref_Deductible1,
    c.ClaimCode AS Ref_ClaimCode,
    c.CoverageInfoGUID AS Ref_CoverageInfoGUID
FROM special_non_origami s
LEFT JOIN coverage_filtered_prod c
  ON c.InTransactionId = s.Ref_InTransactionId
 AND c.ProductId = s.ProductId
 AND c.CoverageCode = s.CoverageCode
"""

OFFSET_SQL["SPECIAL_ORIGAMI_JOINED"] = f"""
SELECT
    so.*,
    NULL AS Ref_RiskItemId,
    NULL AS Ref_ClassCode,
    NULL AS Ref_RiskItemType,
    so.ProductId AS Ref_ProductId,
    c.CoverageId AS Ref_CoverageId,
    c.ParentCoverageId AS Ref_ParentCoverageId,
    c.CoverageExtension AS Ref_CoverageExtension,
    c.CoverageGroup AS Ref_CoverageGroup,
    c.Limit1 AS Ref_Limit1,
    c.Limit2 AS Ref_Limit2,
    c.ASLOB AS Ref_ASLOB,
    c.Option1 AS Option1_ref,
    c.Exposure AS Ref_Exposure,
    c.Deductible1 AS Ref_Deductible1,
    c.ClaimCode AS Ref_ClaimCode,
    c.CoverageInfoGUID AS Ref_CoverageInfoGUID
FROM special_origami so
LEFT JOIN coverage_filtered_prod c
  ON c.InTransactionId = so.Ref_InTransactionId
 AND c.ClaimCode = so.ClaimCode
"""

OFFSET_SQL["NORMAL_FINAL"] = f"""
SELECT
    PolicyNumber,
    PolicyVersion_org AS PolicyVersion,
    Ref_InTransactionId AS InTransactionId,
    ReferencePolicyVersion_org AS ReferencePolicyVersion,
    BookDate,
    ProrataPercent,
    PolicySource,
    LOBCode,
    Ref_RiskItemType AS RiskItemType,
    UnitNumber,
    Ref_ClassCode AS ClassCode,
    PrimaryRiskState,
    Ref_CoverageId AS CoverageId,
    Ref_ParentCoverageId AS ParentCoverageId,
    Ref_RiskItemId AS RiskItemId,
    NULL AS ProductId,
    CoverageCode,
    Ref_CoverageExtension AS CoverageExtension,
    Ref_CoverageGroup AS CoverageGroup,
    CAST(0 AS DECIMAL(18,2)) AS OnsetAmount,
    OffsetAmount,
    Ref_Limit1 AS Limit1,
    Ref_Limit2 AS Limit2,
    Ref_ASLOB AS ASLOB,
    Option1_ref AS Option1,
    Ref_Exposure AS Exposure,
    Ref_Deductible1 AS Deductible1,
    Ref_ClaimCode AS ClaimCode,
    Ref_CoverageInfoGUID AS CoverageInfoGUID
FROM normal_joined
"""

OFFSET_SQL["SPECIAL_FINAL"] = f"""
SELECT
    PolicyNumber,
    PolicyVersion_org AS PolicyVersion,
    Ref_InTransactionId AS InTransactionId,
    ReferencePolicyVersion_org AS ReferencePolicyVersion,
    BookDate,
    ProrataPercent,
    PolicySource,
    LOBCode,
    RiskItemType,
    UnitNumber,
    NULL AS ClassCode,
    PrimaryRiskState,
    Ref_CoverageId AS CoverageId,
    Ref_ParentCoverageId AS ParentCoverageId,
    NULL AS RiskItemId,
    Ref_ProductId AS ProductId,
    CoverageCode,
    Ref_CoverageExtension AS CoverageExtension,
    Ref_CoverageGroup AS CoverageGroup,
    CAST(0 AS DECIMAL(18,2)) AS OnsetAmount,
    OffsetAmount,
    Ref_Limit1 AS Limit1,
    Ref_Limit2 AS Limit2,
    Ref_ASLOB AS ASLOB,
    Option1_ref AS Option1,
    Ref_Exposure AS Exposure,
    Ref_Deductible1 AS Deductible1,
    Ref_ClaimCode AS ClaimCode,
    Ref_CoverageInfoGUID AS CoverageInfoGUID
FROM special_joined
"""


def run_sql_view(sql_key: str, view_name: str) -> DataFrame:
    """Run one offset reference SQL step and register its result as a temp view.

    Args:
        sql_key: Key into OFFSET_SQL.
        view_name: Temp view name used by later steps.

    Returns:
        The resulting DataFrame.
    """
    result = spark.sql(OFFSET_SQL[sql_key])
    result.createOrReplaceTempView(view_name)
    return result


def build_offset_reference_records(merged_df: DataFrame) -> DataFrame:
    """Build offset rows enriched with the coverage details of the referenced transaction.

    An offset (reversal) row refers to an earlier policy version through
    ReferencePolicyVersion. The coverage attributes (limits, ASLOB, deductible,
    claim code and so on) must come from that earlier version, so each offset
    row is re-attached to the coverage of the referenced transaction.

    Flow:
        NORMAL  (has UnitNumber and RiskItemId): match risk item by unit number,
                then coverage by risk item and coverage code.
        SPECIAL (product level): non Origami matches by product and coverage code,
                Origami matches by claim code.

    Args:
        merged_df: Policy data after coverage merging and Origami prorata.

    Returns:
        Offset rows with onset set to zero and reference attributes applied.
    """
    offset_df = (
        merged_df
        .filter(F.col("OffsetAmount").isNotNull() & (F.col("OffsetAmount") != 0))
        .withColumn("OnsetAmount", F.lit(0))
        .withColumn("PolicyVersion_org", F.col("PolicyVersion"))
        .withColumn("InTransactionId_org", F.col("InTransactionId"))
        .withColumn("ReferencePolicyVersion_org", F.col("ReferencePolicyVersion"))
    )
    offset_df.createOrReplaceTempView("df_offset")

    run_sql_view("REF_TRANSACTION", "ref_tx_df")
    run_sql_view("NORMAL_ROWS", "normal_df")
    run_sql_view("SPECIAL_ROWS", "special_df")

    run_sql_view("RISK_ITEM_FIRST_BY_UNIT", "riskitem_filtered")
    run_sql_view("NORMAL_WITH_REFERENCE_RISK", "normal_risk_df")
    run_sql_view("COVERAGE_FIRST_BY_RISK", "coverage_filtered_risk")
    run_sql_view("NORMAL_WITH_REFERENCE_COVERAGE", "normal_joined")

    run_sql_view("COVERAGE_FIRST_BY_PRODUCT", "coverage_filtered_prod")
    run_sql_view("SPECIAL_NON_ORIGAMI", "special_non_origami")
    run_sql_view("SPECIAL_ORIGAMI", "special_origami")
    non_origami_joined = run_sql_view("SPECIAL_NON_ORIGAMI_JOINED", "special_non_origami_joined")
    origami_joined = run_sql_view("SPECIAL_ORIGAMI_JOINED", "special_origami_joined")
    non_origami_joined.unionByName(origami_joined).createOrReplaceTempView("special_joined")

    normal_final = spark.sql(OFFSET_SQL["NORMAL_FINAL"])
    special_final = spark.sql(OFFSET_SQL["SPECIAL_FINAL"])
    return normal_final.unionByName(special_final)


def apply_child_only_rules(df: DataFrame) -> DataFrame:
    """Keep only child coverages for rows matching the child only rules.

    Rows matching a rule are kept when ParentCoverageId is not null. All other
    rows are kept untouched. NULL handling is identical to the original logic:
    rows for which the rule condition evaluates to NULL are dropped.

    Args:
        df: Offset reference rows.

    Returns:
        Offset reference rows after applying the rules.
    """
    if not OFFSET_REFERENCE_CHILD_ONLY_RULES:
        return df
    target_condition = reduce(
        lambda left, right: left | right,
        [build_match_condition(rule) for rule in OFFSET_REFERENCE_CHILD_ONLY_RULES],
    )
    target_children = df.filter(target_condition).filter(F.col("ParentCoverageId").isNotNull())
    rest = df.filter(~target_condition)
    return rest.unionByName(target_children)


def build_onset_rows(merged_df: DataFrame) -> DataFrame:
    """Return onset rows (non zero OnsetAmount) with OffsetAmount forced to zero."""
    return (
        merged_df
        .filter(F.col("OnsetAmount").isNotNull() & (F.col("OnsetAmount") != 0))
        .withColumn("OffsetAmount", F.lit(0))
    )


offset_reference_df = apply_child_only_rules(build_offset_reference_records(final_merged_df))
onset_and_offset_df = build_onset_rows(final_merged_df).unionByName(offset_reference_df)

ONSET_OFFSET_TABLE = ctx.table_name("OnSet_And_Refrence_Offset")
write_delta_table(onset_and_offset_df, ONSET_OFFSET_TABLE)
log(f"Saved {ONSET_OFFSET_TABLE}")

## 7. Claim measures (paid loss, reserves, ALAE)

All claim windows are derived from the reporting period. Liability reserves are valued at the liability reserve month.

In [ ]:
CORRECTION_JOIN_KEYS = [
    "PolicyNumber", "TransactionNumber", "ClaimNumber",
    "ClaimantNumber", "ItemNumber", "DerivedClaimCoverageName",
]

CLAIM_OUTPUT_COLUMNS = [
    "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
    "ClaimCoverageName", "CoverageExtension", "LossDate", "AccidentYear",
    "DerivedClaimCoverageName", "PaidCount", "ReserveCount",
]


def apply_post_period_payment_correction(
    liability_df: DataFrame, correction_df: DataFrame, validate: bool
) -> DataFrame:
    """Add payments and ALAE made after the period end to the liability reserve.

    The correction is aggregated to one row per claim key before the join, so the
    join cannot multiply rows. FinalReserveAmount NULL values become zero.

    Args:
        liability_df: Liability claim measures.
        correction_df: Post period payments by claim key.
        validate: When True, assert that the row count is unchanged by the join.

    Returns:
        Liability claim measures with the corrected FinalReserveAmount.

    Raises:
        RuntimeError: If validation is enabled and the join changed the row count.
    """
    correction_agg = (
        correction_df.groupBy(*CORRECTION_JOIN_KEYS)
        .agg(F.sum(F.coalesce(F.col("PaidAmountALAEPostPeriod"), F.lit(0))).alias("PaidAmountALAEPostPeriod"))
    )

    corrected = (
        liability_df.alias("l")
        .join(correction_agg.alias("a"), on=CORRECTION_JOIN_KEYS, how="left")
        .withColumn(
            "FinalReserveAmount",
            F.coalesce(F.col("l.FinalReserveAmount"), F.lit(0))
            + F.coalesce(F.col("a.PaidAmountALAEPostPeriod"), F.lit(0)),
        )
        .drop("PaidAmountALAEPostPeriod")
    )

    if validate:
        before, after = liability_df.count(), corrected.count()
        log(f"Liability rows before/after payment correction: {before} / {after}")
        if before != after:
            raise RuntimeError(f"Row count changed by payment correction: before={before}, after={after}")
    return corrected


def build_paid_loss_rows(measures_df: DataFrame) -> DataFrame:
    """Build paid loss rows with PaidCount.

    PaidCount is 1 when the period payment is the first payment ever made on the
    claim line, and -1 when the period reverses the full amount paid before.

    Args:
        measures_df: Combined property and liability claim measures.

    Returns:
        Paid loss rows in the claim output layout.
    """
    paid = (
        measures_df
        .filter(F.col("PaymentAmountDuration") != 0)
        .withColumn("PaidLossAmount", F.col("PaymentAmountDuration"))
        .withColumn(
            "PaidCount",
            F.when(
                (F.col("PaymentAmountDuration") == F.col("PaymentAmountDurationAsOfEndOfPeriod"))
                & (F.col("PaymentAmountDuration") > 0)
                & (F.col("PaymentAmountDurationAsOfEndOfPeriod") > 0),
                1,
            )
            .when(
                (F.col("PaymentAmountDuration") < 0)
                & (F.col("PaymentAmountDurationAsOfPriorPeriod") > 0)
                & ((F.col("PaymentAmountDuration") + F.col("PaymentAmountDurationAsOfPriorPeriod")) == 0),
                -1,
            )
            .otherwise(0),
        )
    )
    return paid.select(
        *CLAIM_OUTPUT_COLUMNS,
        "PaidLossAmount",
        F.lit(None).cast("double").alias("ReserveAmount"),
        F.lit(None).cast("double").alias("AllocatedLossAdjExp"),
    )


def build_reserve_rows(measures_df: DataFrame, period: ReportingPeriod) -> DataFrame:
    """Build outstanding reserve rows with ReserveCount.

    Claim lines with an accident date inside the period and no payment or
    incurred loss up to the end of the period are excluded. ReserveCount is 1 when
    the line has incurred loss but no payment yet, and -1 when a reserve that
    existed before the period was fully closed without payment.

    Args:
        measures_df: Combined property and liability claim measures.
        period: Reporting period.

    Returns:
        Reserve rows in the claim output layout.
    """
    accident_ym = F.year("LossDate") * 100 + F.month("LossDate")
    reserve = (
        measures_df
        .filter(
            ~(
                (F.col("PaymentAmountDurationAsOfEndOfPeriod") == 0)
                & (F.col("LossIncurredNOIBNRAsOfEndOfPeriod") == 0)
                & accident_ym.between(period.start_ym, period.end_ym)
            )
        )
        .withColumn("ReserveAmount", F.col("FinalReserveAmount"))
        .filter(F.col("ReserveAmount") != 0)
        .withColumn(
            "ReserveCount",
            F.when(
                (F.col("PaymentAmountDurationAsOfEndOfPeriod") == 0)
                & (F.col("LossIncurredNOIBNRAsOfEndOfPeriod") != 0),
                1,
            )
            .when(
                (F.col("PaymentAmountDurationAsOfEndOfPeriod") == 0)
                & (F.col("LossIncurredNOIBNRAsOfEndOfPeriod") == 0)
                & (F.col("PaymentAmountDurationAsOfPriorPeriod") == 0)
                & (F.col("LossIncurredNOIBNRAsOfPriorPeriod") != 0),
                -1,
            )
            .otherwise(0),
        )
    )
    return reserve.select(
        *CLAIM_OUTPUT_COLUMNS,
        "ReserveAmount",
        F.lit(None).cast("double").alias("PaidLossAmount"),
        F.lit(None).cast("double").alias("AllocatedLossAdjExp"),
    )


def build_allocated_loss_rows(context: RunContext) -> DataFrame:
    """Build ALAE rows paid inside the period."""
    allocated = spark.sql(build_allocated_loss_sql(context.period)).withColumn(
        "TransactionNumber", F.col("TransactionNumber").cast("int")
    )
    return allocated.select(
        *CLAIM_OUTPUT_COLUMNS[:10],
        F.lit(None).cast("integer").alias("PaidCount"),
        F.lit(None).cast("integer").alias("ReserveCount"),
        "AllocatedLossAdjExp",
        F.lit(None).cast("double").alias("ReserveAmount"),
        F.lit(None).cast("double").alias("PaidLossAmount"),
    )


def build_claim_measure_rows(context: RunContext, validate: bool) -> DataFrame:
    """Build paid, reserve and ALAE claim rows for the period.

    Args:
        context: Run context.
        validate: Enable the row count check on the payment correction join.

    Returns:
        Union of paid loss, reserve and ALAE rows.
    """
    liability_df = spark.sql(build_liability_claims_sql(context))
    correction_df = spark.sql(build_post_period_payment_sql(context))
    liability_df = apply_post_period_payment_correction(liability_df, correction_df, validate)

    property_df = spark.sql(build_property_claims_sql(context))

    measures_df = property_df.unionByName(liability_df).withColumn(
        "TransactionNumber", F.col("TransactionNumber").cast("int")
    )

    return reduce(
        lambda left, right: left.unionByName(right),
        [
            build_paid_loss_rows(measures_df),
            build_reserve_rows(measures_df, context.period),
            build_allocated_loss_rows(context),
        ],
    )


def load_claim_guid_mapping() -> DataFrame:
    """Load coverage and risk item GUIDs for claims and fill missing TransactionNumber.

    Some claims have no TransactionNumber in FactClaims. It is filled from
    FactClaimsMonthlyDataStore using policy, claim and claimant.

    Returns:
        GUID mapping with a completed TransactionNumber.
    """
    guid_mapping = spark.sql(f"""
        SELECT PolicyNumber, TransactionNumber, ClaimNumber, ClaimantNumber, ItemNumber,
               ClaimCoverageName, CoverageExtension, CauseOfLoss, ParentCoverageGUID,
               ChildCoverageGUID, ParentRiskitemGUID
        FROM DMAnalytics.FactClaims
        WHERE CompanyCode = '{CLAIMS_COMPANY_CODE}'
          AND LOBCode = '{POLICY_LOB_CODE}'
    """)

    transaction_lookup = (
        spark.table("FactClaimsMonthlyDataStore")
        .select("PolicyNumber", "ClaimNumber", "ClaimantNumber", "TransactionNumber")
        .dropDuplicates(["PolicyNumber", "ClaimNumber", "ClaimantNumber"])
    )

    return (
        guid_mapping.alias("mg")
        .join(
            transaction_lookup.alias("fc"),
            on=[
                F.col("mg.PolicyNumber") == F.col("fc.PolicyNumber"),
                F.col("mg.ClaimNumber") == F.col("fc.ClaimNumber"),
                F.col("mg.ClaimantNumber") == F.col("fc.ClaimantNumber"),
            ],
            how="left",
        )
        .select(
            *[F.col(f"mg.{c}") for c in guid_mapping.columns if c != "TransactionNumber"],
            F.coalesce(F.col("mg.TransactionNumber"), F.col("fc.TransactionNumber")).alias("TransactionNumber"),
        )
    )


def normalize_claim_coverage(claims_df: DataFrame) -> DataFrame:
    """Normalize coverage extension and derived coverage name on claim rows.

    Rules, applied in this order:
        1. BA PLUS on non GMCA policies uses extension CM.
        2. Extensions RR and TL become CM.
        3. MED EXP with extension NF on non GMCA policies becomes PIP LIMIT_MED EXP.
    """
    result = claims_df.withColumn(
        "CoverageExtension",
        F.when(
            (F.col("ClaimCoverageName") == "BA PLUS") & (~F.col("PolicyNumber").startswith("GMCA")),
            "CM",
        ).otherwise(F.col("CoverageExtension")),
    )
    result = result.withColumn(
        "CoverageExtension",
        F.when(F.col("CoverageExtension").isin("RR", "TL"), "CM").otherwise(F.col("CoverageExtension")),
    )
    return result.withColumn(
        "DerivedClaimCoverageName",
        F.when(
            (F.col("DerivedClaimCoverageName") == "MED EXP")
            & (F.col("CoverageExtension") == "NF")
            & (~F.col("PolicyNumber").startswith("GMCA")),
            "PIP LIMIT_MED EXP",
        ).otherwise(F.col("DerivedClaimCoverageName")),
    )


def attach_claim_guids(claims_df: DataFrame, guid_mapping: DataFrame) -> DataFrame:
    """Attach coverage and risk item GUIDs to claim rows.

    The mapping is de-duplicated first so the left join cannot multiply rows.
    TransactionNumber and ItemNumber are cast to int for the comparison only.

    Args:
        claims_df: Normalized claim rows.
        guid_mapping: Output of load_claim_guid_mapping.

    Returns:
        Claim rows plus ParentCoverageGUID, ChildCoverageGUID, CauseOfLoss and
        ParentRiskitemGUID.
    """
    mapping_trimmed = (
        guid_mapping
        .dropDuplicates([
            "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber",
            "ItemNumber", "ClaimCoverageName", "CoverageExtension", "ParentRiskitemGUID",
        ])
        .select(
            "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
            "ClaimCoverageName", "CoverageExtension", "ParentCoverageGUID", "ChildCoverageGUID",
            "CauseOfLoss", "ParentRiskitemGUID",
        )
    )

    join_condition = (
        (F.col("f.PolicyNumber") == F.col("m.PolicyNumber"))
        & (F.col("f.ClaimNumber") == F.col("m.ClaimNumber"))
        & (F.col("f.ClaimantNumber") == F.col("m.ClaimantNumber"))
        & (F.col("f.ItemNumber").cast("int") == F.col("m.ItemNumber").cast("int"))
        & (F.col("f.DerivedClaimCoverageName") == F.col("m.ClaimCoverageName"))
        & (F.col("f.CoverageExtension") == F.col("m.CoverageExtension"))
        & (F.col("f.TransactionNumber").cast("int") == F.col("m.TransactionNumber").cast("int"))
    )

    return (
        claims_df.alias("f")
        .join(mapping_trimmed.alias("m"), on=join_condition, how="left")
        .select(
            "f.*",
            "m.ParentCoverageGUID", "m.ChildCoverageGUID", "m.CauseOfLoss", "m.ParentRiskitemGUID",
        )
    )


claim_measure_rows = build_claim_measure_rows(ctx, RUN_VALIDATIONS)
claim_output_df = attach_claim_guids(
    normalize_claim_coverage(claim_measure_rows),
    load_claim_guid_mapping(),
)
log("Claim measures defined (lazy). Next: map claims to policy coverages.")

## Legacy behaviors preserved

The following behaviors exist in the original yearly notebook. They are kept exactly so that yearly results do not change. They are listed here so the business can confirm or change them deliberately.

1. **Paid loss and ALAE rows are removed before the claim to policy mapping.** The original filter `ReserveAmount >= 0` also removes rows where `ReserveAmount` is NULL, and paid loss and ALAE rows always have a NULL reserve. In effect only reserve rows reach the final claim output. This is controlled by `KEEP_NON_RESERVE_CLAIM_ROWS` (default `False` = legacy behavior). Set it to `True` only after the business confirms that paid loss and ALAE must appear in the report.
2. **Liability reserves can be valued later than the period end** (the original used March 2026 for the 2025 run), while property reserves are valued at the period end. This is now the parameter `liability_reserve_as_of`.
3. **Coverage rows with a NULL ClaimCode and a parent coverage are not reported.** The parent join in the NISS mapping stage was computed but never combined with the child join in the original, so only the child join result is used.
4. **ReserveAmount is a period end balance, not a movement.** A monthly run reports the reserve balance at the end of that month, exactly as the yearly run reports the balance at the end of the year. Flow measures (premium, exposure, paid loss, ALAE, paid counts) are additive across months. Reserve balances are not.

## 8. Claim to policy mapping

Claims are linked to policy coverages through the coverage and risk item GUIDs, with fallbacks for unmatched GUIDs. The result is saved to `Claim_Policy_NewApproachData`.

In [ ]:
def filter_claims_by_term_effective_date(claims_df: DataFrame) -> DataFrame:
    """Keep claims whose policy term started on or after the cutoff date.

    Args:
        claims_df: Claim rows with PolicyNumber and TransactionNumber.

    Returns:
        Claim rows in scope (claim columns only).
    """
    policy_detail = spark.read.table(f"{ODS_SCHEMA}.policy_policy_detail_info")
    return (
        claims_df.alias("c")
        .join(
            policy_detail.alias("p"),
            (F.col("c.PolicyNumber") == F.col("p.PolicyNumber"))
            & (F.col("c.TransactionNumber").cast("int") == F.col("p.PolicyVersion").cast("int")),
            "left",
        )
        .filter(F.col("p.TermEffectiveDate") >= F.lit(TERM_EFFECTIVE_CUTOFF.isoformat()))
        .select("c.*")
    )


def filter_claim_reserve_rows(claims_df: DataFrame, keep_non_reserve_rows: bool) -> DataFrame:
    """Remove negative reserve rows (see the Legacy behaviors preserved cell).

    Args:
        claims_df: Claim rows in scope.
        keep_non_reserve_rows: False reproduces the original filter, which also
            removes rows whose ReserveAmount is NULL (paid loss and ALAE rows).

    Returns:
        Filtered claim rows.
    """
    if keep_non_reserve_rows:
        return claims_df.filter(F.col("ReserveAmount").isNull() | (F.col("ReserveAmount") >= 0))
    return claims_df.filter(F.col("ReserveAmount") >= 0)


def enrich_claims_with_policy(claims_df: DataFrame) -> DataFrame:
    """Attach policy, product, risk item and coverage details to claims through GUIDs.

    Args:
        claims_df: Claim rows in scope.

    Returns:
        Claims plus policy attributes. Policy coverage extension and GUID are
        exposed as PolicyCoverageExtension and PolicyCoverageInfoGUID.
    """
    pdi = spark.table(f"{ODS_SCHEMA}.policy_policy_detail_info").alias("pdi")
    pi = spark.table(f"{ODS_SCHEMA}.policy_Product_info").alias("pi")
    ri = spark.table(f"{ODS_SCHEMA}.policy_risk_item_info").alias("ri")
    ci = spark.table(f"{ODS_SCHEMA}.policy_coverage_info").alias("ci")

    joined = (
        claims_df.alias("c")
        .join(
            pdi,
            (F.col("c.PolicyNumber") == F.col("pdi.PolicyNumber"))
            & (F.col("c.TransactionNumber").cast("int") == F.col("pdi.PolicyVersion").cast("int")),
            "left",
        )
        .join(pi, F.col("pdi.InTransactionId") == F.col("pi.InTransactionId"), "left")
        .join(
            ri,
            (F.col("pdi.InTransactionId") == F.col("ri.InTransactionId"))
            & (F.upper(F.col("c.ParentRiskitemGUID")) == F.upper(F.col("ri.RiskItemInfoGUID"))),
            "left",
        )
        .join(
            ci,
            (F.col("pdi.InTransactionId") == F.col("ci.InTransactionId"))
            & (F.upper(F.col("c.ParentCoverageGUID")) == F.upper(F.col("ci.CoverageInfoGUID"))),
            "left",
        )
    )

    return joined.select(
        F.col("c.*"),
        F.col("pdi.PolicyVersion"), F.col("pdi.InTransactionId"), F.col("pdi.ReferencePolicyVersion"),
        F.col("pdi.PolicySource"), F.col("pdi.BookDate"), F.col("pdi.PrimaryRiskState"),
        F.col("pdi.ProrataPercent"),
        F.col("pi.LOBCode"),
        F.col("ri.RiskItemType"), F.col("ri.UnitNumber"), F.col("ri.ClassCode"),
        F.col("ci.CoverageId"), F.col("ci.ParentCoverageId"), F.col("ci.RiskItemId"), F.col("ci.ProductId"),
        F.col("ci.CoverageCode"),
        F.col("ci.CoverageExtension").alias("PolicyCoverageExtension"),
        F.col("ci.CoverageGroup"), F.col("ci.OnsetAmount"), F.col("ci.OffsetAmount"),
        F.col("ci.Limit1"), F.col("ci.Limit2"), F.col("ci.ASLOB"), F.col("ci.Option1"),
        F.col("ci.Exposure"), F.col("ci.Deductible1"), F.col("ci.ClaimCode"),
        F.col("ci.CoverageInfoGUID").alias("PolicyCoverageInfoGUID"),
    )


def find_claims_with_unmatched_guids(claims_df: DataFrame) -> DataFrame:
    """Return claim rows whose coverage or risk item GUID does not match the policy.

    A claim row is classified as MISSING_IN_CLAIM, MISSING_IN_POLICY,
    AVAILABLE_BUT_NOT_MATCHED or MATCHED. Only rows that are not MATCHED are returned.

    Args:
        claims_df: Claim rows in scope.

    Returns:
        Unmatched claim rows with the validation status columns.
    """
    claim_df = (
        claims_df
        .withColumn("ParentCoverageGUID", F.upper(F.trim(F.col("ParentCoverageGUID"))))
        .withColumn("ParentRiskitemGUID", F.upper(F.trim(F.col("ParentRiskitemGUID"))))
        .alias("c")
    )
    coverage_guids = (
        spark.table(f"{ODS_SCHEMA}.policy_coverage_info")
        .select(F.upper(F.trim(F.col("CoverageInfoGUID"))).alias("PolicyCoverageGUID"))
        .distinct()
    )
    risk_item_guids = (
        spark.table(f"{ODS_SCHEMA}.policy_risk_item_info")
        .select(F.upper(F.trim(F.col("RiskItemInfoGUID"))).alias("PolicyRiskItemGUID"))
        .distinct()
    )

    debug_df = (
        claim_df
        .join(coverage_guids, F.col("ParentCoverageGUID") == F.col("PolicyCoverageGUID"), "left")
        .join(risk_item_guids, F.col("ParentRiskitemGUID") == F.col("PolicyRiskItemGUID"), "left")
    )

    debug_df = debug_df.withColumn(
        "ValidationStatus",
        F.when(F.col("ParentCoverageGUID").isNull() | F.col("ParentRiskitemGUID").isNull(), "MISSING_IN_CLAIM")
        .when(
            (F.col("ParentCoverageGUID").isNotNull() & F.col("PolicyCoverageGUID").isNull())
            | (F.col("ParentRiskitemGUID").isNotNull() & F.col("PolicyRiskItemGUID").isNull()),
            "MISSING_IN_POLICY",
        )
        .when(
            F.col("PolicyCoverageGUID").isNotNull()
            & F.col("PolicyRiskItemGUID").isNotNull()
            & (
                (F.col("ParentCoverageGUID") != F.col("PolicyCoverageGUID"))
                | (F.col("ParentRiskitemGUID") != F.col("PolicyRiskItemGUID"))
            ),
            "AVAILABLE_BUT_NOT_MATCHED",
        )
        .otherwise("MATCHED"),
    )
    debug_df = (
        debug_df
        .withColumn(
            "CoverageStatus",
            F.when(F.col("ParentCoverageGUID").isNull(), "MISSING_IN_CLAIM")
            .when(F.col("PolicyCoverageGUID").isNull(), "MISSING_IN_POLICY")
            .otherwise("AVAILABLE"),
        )
        .withColumn(
            "RiskItemStatus",
            F.when(F.col("ParentRiskitemGUID").isNull(), "MISSING_IN_CLAIM")
            .when(F.col("PolicyRiskItemGUID").isNull(), "MISSING_IN_POLICY")
            .otherwise("AVAILABLE"),
        )
    )
    return debug_df.filter(F.col("ValidationStatus") != "MATCHED")


def build_policy_unit_lookup() -> DataFrame:
    """Policy coverages keyed by the printed unit number (ItemNumber).

    One row is kept per policy version, item number, coverage code and coverage
    extension, so joining it to claims cannot multiply rows.
    """
    pdi = spark.table(f"{ODS_SCHEMA}.policy_policy_detail_info")
    ri = spark.table(f"{ODS_SCHEMA}.policy_risk_item_info")
    ci = spark.table(f"{ODS_SCHEMA}.policy_coverage_info")
    qa = spark.table(f"{ODS_SCHEMA}.policy_question_answer")

    print_unit = (
        qa.filter(F.col("QuestionCode") == "PrintUnitNumber")
        .select("InTransactionId", "RiskItemId", F.col("TextValue").alias("ItemNumber"))
    )

    policy_rows = (
        pdi.alias("pdi")
        .join(ri.alias("ri"), F.col("pdi.InTransactionId") == F.col("ri.InTransactionId"))
        .join(
            ci.alias("ci"),
            (F.col("pdi.InTransactionId") == F.col("ci.InTransactionId"))
            & (F.col("ri.RiskItemId") == F.col("ci.RiskItemId")),
        )
        .join(
            print_unit.alias("qa"),
            (F.col("pdi.InTransactionId") == F.col("qa.InTransactionId"))
            & (F.col("ri.RiskItemId") == F.col("qa.RiskItemId")),
        )
        .select(
            F.col("pdi.InTransactionId"), F.col("pdi.PolicyNumber"), F.col("pdi.PolicyVersion"),
            F.col("pdi.ReferencePolicyVersion"), F.col("pdi.PolicySource"), F.col("pdi.BookDate"),
            F.col("pdi.PrimaryRiskState"), F.col("pdi.ProrataPercent"), F.col("qa.ItemNumber"),
            F.col("ci.CoverageCode"), F.col("ci.CoverageExtension").alias("PolicyCoverageExtension"),
            F.col("ri.RiskItemType"), F.col("ri.UnitNumber"), F.col("ri.ClassCode"),
            F.col("ci.CoverageId"), F.col("ci.ParentCoverageId"), F.col("ci.RiskItemId"),
            F.col("ci.ProductId"), F.col("ci.CoverageGroup"), F.col("ci.OnsetAmount"),
            F.col("ci.OffsetAmount"), F.col("ci.Limit1"), F.col("ci.Limit2"), F.col("ci.ASLOB"),
            F.col("ci.Option1"), F.col("ci.Exposure"), F.col("ci.Deductible1"), F.col("ci.ClaimCode"),
            F.col("ci.CoverageInfoGUID"),
        )
    )

    window_spec = Window.partitionBy(
        "PolicyNumber", "PolicyVersion", "ItemNumber", "CoverageCode", "PolicyCoverageExtension"
    ).orderBy(F.col("CoverageId"))
    return (
        policy_rows.withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )


def recover_by_item_number(unmatched_df: DataFrame) -> DataFrame:
    """Recover policy coverage for unmatched claims that carry an item number.

    Matches on policy, policy version, item number, derived coverage name and
    coverage extension.
    """
    items = unmatched_df.filter(F.col("ItemNumber") != 0)
    lookup = build_policy_unit_lookup()
    return (
        items.alias("udf")
        .join(
            lookup.alias("pol"),
            (F.col("udf.PolicyNumber") == F.col("pol.PolicyNumber"))
            & (F.col("udf.TransactionNumber").cast("int") == F.col("pol.PolicyVersion").cast("int"))
            & (F.col("udf.ItemNumber").cast("int") == F.col("pol.ItemNumber").cast("int"))
            & (F.col("udf.DerivedClaimCoverageName") == F.col("pol.CoverageCode"))
            & (F.col("udf.CoverageExtension") == F.col("pol.PolicyCoverageExtension")),
            "left",
        )
        .select(
            "udf.*",
            F.col("pol.InTransactionId"), F.col("pol.PolicyVersion"), F.col("pol.ReferencePolicyVersion"),
            F.col("pol.PolicySource"), F.col("pol.BookDate"),
            F.lit(POLICY_LOB_CODE).alias("LOBCode"),
            F.col("pol.PrimaryRiskState"), F.col("pol.ProrataPercent"),
            F.col("pol.RiskItemType"), F.col("pol.UnitNumber"), F.col("pol.CoverageCode"),
            F.col("pol.PolicyCoverageExtension"), F.col("pol.ClassCode"), F.col("pol.CoverageId"),
            F.col("pol.ParentCoverageId"), F.col("pol.RiskItemId"), F.col("pol.ProductId"),
            F.col("pol.CoverageGroup"), F.col("pol.OnsetAmount"), F.col("pol.OffsetAmount"),
            F.col("pol.Limit1"), F.col("pol.Limit2"), F.col("pol.ASLOB"), F.col("pol.Option1"),
            F.col("pol.Exposure"), F.col("pol.Deductible1"), F.col("pol.ClaimCode"),
            F.col("pol.CoverageInfoGUID").alias("PolicyCoverageInfoGUID"),
        )
    )


def recover_by_claim_coverage(unmatched_df: DataFrame) -> DataFrame:
    """Recover policy coverage for unmatched claims without an item number.

    Matches the policy transaction, then the coverage whose CoverageCode equals
    the claim coverage name. Risk item fields stay empty.
    """
    zero_items = unmatched_df.filter(F.col("ItemNumber") == 0)
    pdi = spark.table(f"{ODS_SCHEMA}.policy_policy_detail_info")
    ci = spark.table(f"{ODS_SCHEMA}.policy_coverage_info")

    policy_base = (
        zero_items.alias("udf")
        .join(
            pdi.alias("pdi"),
            (F.col("udf.PolicyNumber") == F.col("pdi.PolicyNumber"))
            & (F.col("udf.TransactionNumber") == F.col("pdi.PolicyVersion")),
            "left",
        )
        .select(
            "udf.*",
            F.col("pdi.InTransactionId"), F.col("pdi.PolicySource"), F.col("pdi.PolicyVersion"),
            F.col("pdi.ReferencePolicyVersion"), F.col("pdi.BookDate"), F.col("pdi.ProrataPercent"),
            F.col("pdi.PrimaryRiskState"),
        )
    )
    return (
        policy_base.alias("base")
        .join(
            ci.alias("ci"),
            (F.col("base.InTransactionId") == F.col("ci.InTransactionId"))
            & (F.col("base.ClaimCoverageName") == F.col("ci.CoverageCode")),
            "left",
        )
        .select(
            "base.*",
            F.lit(POLICY_LOB_CODE).alias("LOBCode"),
            F.lit(None).alias("RiskItemType"), F.lit(None).alias("UnitNumber"), F.lit(None).alias("ClassCode"),
            F.col("ci.CoverageId"), F.col("ci.ParentCoverageId"), F.col("ci.RiskItemId"), F.col("ci.ProductId"),
            F.col("ci.CoverageCode"), F.col("ci.CoverageExtension").alias("PolicyCoverageExtension"),
            F.col("ci.CoverageGroup"),
            F.lit(None).cast("double").alias("OnsetAmount"),
            F.lit(None).cast("double").alias("OffsetAmount"),
            F.col("ci.Limit1"), F.col("ci.Limit2"), F.col("ci.ASLOB"), F.col("ci.Option1"),
            F.col("ci.Exposure"), F.col("ci.Deductible1"), F.col("ci.ClaimCode"),
            F.col("ci.CoverageInfoGUID").alias("PolicyCoverageInfoGUID"),
        )
    )


FALLBACK_OUTPUT_COLUMNS = [
    "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber", "ItemNumber",
    "ClaimCoverageName", "CoverageExtension", "LossDate", "AccidentYear", "DerivedClaimCoverageName",
    "PaidCount", "ReserveCount", "PaidLossAmount", "ReserveAmount", "AllocatedLossAdjExp",
    "ParentCoverageGUID", "ChildCoverageGUID", "CauseOfLoss", "ParentRiskitemGUID",
    "PolicyVersion", "InTransactionId", "ReferencePolicyVersion", "PolicySource", "BookDate",
]
FALLBACK_POLICY_COLUMNS = [
    "PrimaryRiskState", "LOBCode", "RiskItemType", "UnitNumber", "ClassCode", "CoverageId",
    "ParentCoverageId", "RiskItemId", "ProductId", "CoverageCode", "PolicyCoverageExtension",
    "CoverageGroup",
]
FALLBACK_TAIL_COLUMNS = [
    "Limit1", "Limit2", "ASLOB", "Option1", "Deductible1", "ClaimCode", "PolicyCoverageInfoGUID",
]


def build_claim_policy_data(claims_df: DataFrame) -> DataFrame:
    """Build claim rows enriched with policy coverage details.

    Steps:
        1. Map claims to policy coverages through the coverage and risk item GUIDs.
        2. For claims whose GUIDs do not match, recover the policy coverage by
           item number, or by claim coverage name when there is no item number.
        3. Replace the original rows of the recovered claims with the recovered rows.

    Args:
        claims_df: Claim rows in scope after the term and reserve filters.

    Returns:
        Claim and policy data using the final column names (CoverageExtension is
        the policy coverage extension, ClaimCoverageExtension is the claim one).
    """
    enriched_df = enrich_claims_with_policy(claims_df)

    unmatched_df = apply_override_rules(
        find_claims_with_unmatched_guids(claims_df),
        "DerivedClaimCoverageName",
        DERIVED_COVERAGE_NAME_OVERRIDES,
    )

    recovered_df = recover_by_item_number(unmatched_df).unionByName(recover_by_claim_coverage(unmatched_df))
    recovered_df = recovered_df.select(
        *FALLBACK_OUTPUT_COLUMNS,
        F.lit(None).alias("ProrataPercent"),
        *FALLBACK_POLICY_COLUMNS,
        F.lit(None).cast("double").alias("OnsetAmount"),
        F.lit(None).cast("double").alias("OffsetAmount"),
        "Limit1", "Limit2", "ASLOB", "Option1",
        F.lit(None).alias("Exposure"),
        "Deductible1", "ClaimCode", "PolicyCoverageInfoGUID",
    )

    replace_keys = [
        "PolicyNumber", "TransactionNumber", "ClaimNumber", "ClaimantNumber",
        "ClaimCoverageName", "CoverageExtension",
    ]
    kept_original = enriched_df.alias("orig").join(
        recovered_df.select(*replace_keys).distinct().alias("fix"), on=replace_keys, how="leftanti"
    )
    combined = kept_original.unionByName(recovered_df)

    return (
        combined
        .withColumnRenamed("CoverageExtension", "ClaimCoverageExtension")
        .withColumnRenamed("PolicyCoverageInfoGUID", "CoverageInfoGUID")
        .withColumnRenamed("PolicyCoverageExtension", "CoverageExtension")
    )


claims_in_scope = filter_claim_reserve_rows(
    filter_claims_by_term_effective_date(claim_output_df),
    KEEP_NON_RESERVE_CLAIM_ROWS,
)
if not KEEP_NON_RESERVE_CLAIM_ROWS:
    log("Legacy claim filter is active: claim rows with a NULL ReserveAmount "
        "(paid loss and ALAE rows) are excluded from the report.")

claim_policy_df = build_claim_policy_data(claims_in_scope)

CLAIM_POLICY_TABLE = ctx.table_name("Claim_Policy_NewApproachData")
write_delta_table(claim_policy_df, CLAIM_POLICY_TABLE)
log(f"Saved {CLAIM_POLICY_TABLE}")

## 9. Combine policy and claim rows and attach ZIP codes

Saved to `OnSet_And_Refrence_Offset_Address`.

In [ ]:
def combine_policy_and_claim_data() -> DataFrame:
    """Union policy (onset and offset) rows with claim rows.

    Claim rows carry no premium, so OnsetAmount and OffsetAmount are set to zero.
    Exposure and ProrataPercent are empty on claim rows.

    Returns:
        Combined data. Columns missing on one side are filled with NULL.
    """
    policy_rows = spark.table(ONSET_OFFSET_TABLE)
    claim_rows = (
        spark.table(CLAIM_POLICY_TABLE)
        .withColumn("OffsetAmount", F.lit(0))
        .withColumn("OnsetAmount", F.lit(0))
        .withColumn("Exposure", F.lit(None))
        .withColumn("ProrataPercent", F.lit(None))
    )
    return policy_rows.unionByName(claim_rows, allowMissingColumns=True)


def build_ibmi_zip_lookup(data: DataFrame) -> DataFrame:
    """ZIP code per transaction for IBMi policies.

    The ZIP comes from the address of the vehicle risk item whose LocationNumber
    answer is 1. One row is kept per transaction.
    """
    transactions = data.filter("PolicySource = 'IBMi'").select("InTransactionId").distinct()

    vehicle_items = spark.sql(f"""
        SELECT InTransactionId, RiskItemId
        FROM {ODS_SCHEMA}.policy_risk_item_info
        WHERE RiskItemType = 'Vehicle'
    """).join(transactions, on="InTransactionId", how="inner")

    location_answers = spark.sql(f"""
        SELECT InTransactionId, RiskItemId, TextValue
        FROM {ODS_SCHEMA}.policy_question_answer
        WHERE QuestionCode = 'LocationNumber'
    """)
    location_one = (
        vehicle_items.join(location_answers, ["InTransactionId", "RiskItemId"], "inner")
        .filter("TextValue = '1'")
    )

    addresses = spark.sql(f"""
        SELECT PA.InTransactionId, PA.RiskItemId, P.ZipCode, P.ZipExt
        FROM {ODS_SCHEMA}.policy_policy_address PA
        JOIN {ODS_SCHEMA}.party_address P
          ON P.AddressId = PA.AddressID
    """)

    return (
        location_one.join(addresses, ["InTransactionId", "RiskItemId"], "inner")
        .select("InTransactionId", "ZipCode", "ZipExt")
        .groupBy("InTransactionId")
        .agg(
            F.first("ZipCode", ignorenulls=True).alias("ZipCode"),
            F.first("ZipExt", ignorenulls=True).alias("ZipExt"),
        )
    )


def build_origami_zip_lookup(data: DataFrame) -> DataFrame:
    """ZIP code per transaction for Origami policies.

    The ZIP comes from the risk, garage or other garage address of the vehicle
    risk items. The first non blank ZIP per transaction is kept.
    """
    transactions = data.filter("PolicySource = 'Origami'").select("InTransactionId").distinct()

    vehicle_items = spark.sql(f"""
        SELECT InTransactionId, RiskItemId
        FROM {ODS_SCHEMA}.policy_risk_item_info
        WHERE RiskItemType = 'Vehicle'
    """).join(transactions, "InTransactionId", "inner")

    addresses = spark.sql(f"""
        SELECT PA.InTransactionId, PA.RiskItemId, P.ZipCode, P.ZipExt
        FROM {ODS_SCHEMA}.policy_policy_address PA
        JOIN {ODS_SCHEMA}.party_address P
          ON P.AddressId = PA.AddressID
        WHERE PA.AddressUse IN (
            'Risk Address',
            'Garage Address-PolicyLocation',
            'Garage Address - Other',
            'Garage Address-Other'
        )
    """)

    return (
        addresses.join(vehicle_items, ["InTransactionId", "RiskItemId"], "inner")
        .filter(F.col("ZipCode").isNotNull() & (F.trim(F.col("ZipCode")) != ""))
        .groupBy("InTransactionId")
        .agg(
            F.first(F.col("ZipCode"), ignorenulls=True).alias("ZipCode"),
            F.first(F.col("ZipExt"), ignorenulls=True).alias("ZipExt"),
        )
    )


def attach_zip_codes(data: DataFrame) -> DataFrame:
    """Attach ZipCode and ZipExt (primary risk address) to every row.

    IBMi ZIP codes are applied first. Origami ZIP codes fill rows that still have
    no ZIP.

    Args:
        data: Combined policy and claim data.

    Returns:
        Data with ZipCode and ZipExt.
    """
    with_ibmi_zip = data.join(build_ibmi_zip_lookup(data), on="InTransactionId", how="left")

    origami_zip = build_origami_zip_lookup(data)
    base_columns = [c for c in with_ibmi_zip.columns if c not in ("ZipCode", "ZipExt")]
    return (
        with_ibmi_zip.alias("base")
        .join(origami_zip.alias("org"), on="InTransactionId", how="left")
        .select(
            *[F.col(f"base.{c}") for c in base_columns],
            F.coalesce(F.col("base.ZipCode"), F.col("org.ZipCode")).alias("ZipCode"),
            F.coalesce(F.col("base.ZipExt"), F.col("org.ZipExt")).alias("ZipExt"),
        )
    )


combined_df = combine_policy_and_claim_data()
address_df = attach_zip_codes(combined_df)

ADDRESS_TABLE = ctx.table_name("OnSet_And_Refrence_Offset_Address")
write_delta_table(address_df, ADDRESS_TABLE)
log(f"Saved {ADDRESS_TABLE}")

## 10. STAT rules, ASLOB, ZIP corrections and territory code

In [ ]:
def case_when_map(key: Column, mapping: Dict[str, object], condition: Column, default: Column) -> Column:
    """Build a CASE WHEN chain: when condition and key equals a mapping key, use its value.

    Args:
        key: Column compared with the mapping keys.
        mapping: Key to value dictionary.
        condition: Extra condition that must hold for every branch.
        default: Value used when no branch matches.

    Returns:
        A Spark Column expression.
    """
    expression = None
    for map_key, map_value in mapping.items():
        branch = condition & (key == map_key)
        expression = F.when(branch, F.lit(map_value)) if expression is None else expression.when(branch, F.lit(map_value))
    return expression.otherwise(default)


# IBMi rules (STAT 83 update)
IBMI_COVERAGE_EXTENSION = {
    "UPD DED": "UPD", "PIP LIMIT": "NF", "BI": "CBI", "BA PLUS": "CM", "BROAD PIP": "BI",
}
IBMI_CBI_TO_BI_COVERAGES = ["FARMTR", "FRMTRKPOLL", "HIRED AUTO", "NONOWNAUTO", "DOC"]
IBMI_COVERAGE_GROUP = {"BA PLUS": "Property", "UPD DED": "Liability"}
IBMI_ASLOB = {
    "BA PLUS": 212, "BROAD PIP": 194, "BI": 194, "BLNKT AI": 194, "DOC": 194, "FARMTR": 194,
    "FRMTRKPOLL": 194, "GK COLL": 212, "GK COMP": 212, "HIRED AUTO": 194, "HIRED PHYS": 212,
    "NONOWNAUTO": 194, "PIP LIMIT": 193, "UBI HIRED": 194, "UBI NONOWN": 194,
    "UIM HIRED": 194, "UIM NONOWN": 194, "UPD DED": 194,
}
# Origami default ASLOB, used only when ASLOB is still empty.
ORIGAMI_DEFAULT_ASLOB = {
    "BODINJLIAB": 194, "COLLISION": 212, "UIM": 194, "COMB-FPB": 193,
    "MED EXP": 193, "OTC": 212, "UBI": 194,
}


def apply_stat83_coverage_rules(df: DataFrame) -> DataFrame:
    """Apply coverage extension, coverage group and ASLOB rules by policy source.

    Args:
        df: Combined policy and claim data with ZIP codes.

    Returns:
        Data with corrected CoverageExtension, CoverageGroup and ASLOB.
    """
    is_ibmi = F.col("PolicySource") == "IBMi"
    is_origami = F.col("PolicySource") == "Origami"
    coverage_code = F.col("CoverageCode")

    # Explicit IBMi mapping first, then the CBI to BI rule, otherwise keep the value.
    cbi_to_bi = F.when(
        is_ibmi & coverage_code.isin(*IBMI_CBI_TO_BI_COVERAGES) & (F.col("CoverageExtension") == "CBI"), "BI"
    ).otherwise(F.col("CoverageExtension"))
    extension = case_when_map(coverage_code, IBMI_COVERAGE_EXTENSION, is_ibmi, cbi_to_bi)
    df = df.withColumn("CoverageExtension", extension)

    df = df.withColumn(
        "CoverageExtension",
        F.when(is_origami & (coverage_code == "PIP LIMIT"), "NF").otherwise(F.col("CoverageExtension")),
    )

    df = df.withColumn(
        "CoverageGroup",
        case_when_map(coverage_code, IBMI_COVERAGE_GROUP, is_ibmi, F.col("CoverageGroup")),
    )

    return df.withColumn("ASLOB", case_when_map(coverage_code, IBMI_ASLOB, is_ibmi, F.col("ASLOB")))


def apply_aslob_manual_updates(df: DataFrame) -> DataFrame:
    """Fill empty ASLOB from the manual lookup, then Origami defaults, then BA PLUS rules.

    Args:
        df: Data after the STAT 83 rules.

    Returns:
        Data with ASLOB completed.
    """
    join_columns = ["PolicySource", "CoverageCode", "CoverageExtension"]
    manual = read_lookup_csv(LOOKUP_FILES["aslob_manual_update"]).select(
        *join_columns, F.col("Correct ASLOB").alias("ASLOB_new")
    )
    joined = df.alias("f").join(manual.alias("m"), on=join_columns, how="left")
    df = joined.select(
        *[
            F.when(F.col("f.ASLOB").isNull(), F.col("m.ASLOB_new")).otherwise(F.col("f.ASLOB")).alias("ASLOB")
            if c == "ASLOB" else F.col(f"f.{c}")
            for c in df.columns
        ]
    )

    df = df.withColumn(
        "ASLOB",
        case_when_map(
            F.col("CoverageCode"),
            ORIGAMI_DEFAULT_ASLOB,
            F.col("ASLOB").isNull() & (F.col("PolicySource") == "Origami"),
            F.col("ASLOB"),
        ),
    )

    # BA PLUS policy rows listed in the lookup (no claim attached) use ASLOB 194.
    ba_plus_lookup = (
        read_lookup_csv(LOOKUP_FILES["ba_plus_aslob"])
        .select("PolicyNumber", "PolicyVersion", "CoverageCode")
        .dropDuplicates()
        .withColumn("match_flag", F.lit(1))
    )
    return (
        df.join(ba_plus_lookup, on=["PolicyNumber", "PolicyVersion", "CoverageCode"], how="left")
        .withColumn(
            "ASLOB",
            F.when((F.col("match_flag") == 1) & F.col("ClaimNumber").isNull(), F.lit(194)).otherwise(F.col("ASLOB")),
        )
        .drop("match_flag")
    )


def apply_zip_corrections(df: DataFrame) -> DataFrame:
    """Fill and correct ZIP codes from lookups and the manual override register.

    Order: fill empty ZIP from the missing ZIP lookup, apply the override register,
    then overwrite with the transaction and coverage level ZIP fix lookup.

    Args:
        df: Data with ZipCode.

    Returns:
        Data with corrected ZipCode.
    """
    missing_zip = (
        read_lookup_csv(LOOKUP_FILES["missing_zip"])
        .select(F.col("InTransactionId").alias("ZIP_InTransactionId"), F.col("ZIP Code").alias("ZIP_From_Manual"))
        .dropDuplicates(["ZIP_InTransactionId"])
    )
    filled = (
        df.join(missing_zip, df["InTransactionId"] == missing_zip["ZIP_InTransactionId"], "left")
        .withColumn("ZipCode", F.when(F.col("ZipCode").isNull(), F.col("ZIP_From_Manual")).otherwise(F.col("ZipCode")))
        .drop("ZIP_InTransactionId", "ZIP_From_Manual")
    )

    filled = apply_override_rules(filled, "ZipCode", ZIP_CODE_OVERRIDES)

    zip_fix = (
        read_lookup_csv(LOOKUP_FILES["zip_further_fix"])
        .select(
            F.col("InTransactionId").alias("FIX_InTransactionId"),
            F.col("CoverageId").alias("FIX_CoverageId"),
            F.col("ZIP Code").alias("ZIP_Fix_Value"),
        )
        .dropDuplicates(["FIX_InTransactionId", "FIX_CoverageId"])
    )
    return (
        filled.join(
            zip_fix,
            (filled["InTransactionId"] == zip_fix["FIX_InTransactionId"])
            & (filled["CoverageId"] == zip_fix["FIX_CoverageId"]),
            "left",
        )
        .withColumn(
            "ZipCode",
            F.when(F.col("ZIP_Fix_Value").isNotNull(), F.col("ZIP_Fix_Value")).otherwise(F.col("ZipCode")),
        )
        .drop("FIX_InTransactionId", "FIX_CoverageId", "ZIP_Fix_Value")
    )


def attach_territory_code(df: DataFrame, territory_file: str) -> DataFrame:
    """Attach TerritoryCode from the territory lookup for the supported states.

    Args:
        df: Data with ZipCode and PrimaryRiskState (state abbreviation).
        territory_file: Territory lookup CSV path.

    Returns:
        Data with TerritoryCode.
    """
    territory = (
        read_lookup_csv(territory_file)
        .filter(F.col("StateAbbrev").isin(ALLOWED_TERRITORY_STATES))
        .select("ZIP", "StateAbbrev", "Territory")
    )
    return (
        df.alias("f")
        .join(
            territory.alias("t"),
            (F.col("f.ZipCode") == F.col("t.ZIP")) & (F.col("f.PrimaryRiskState") == F.col("t.StateAbbrev")),
            how="left",
        )
        .withColumn(
            "TerritoryCode",
            F.when(F.col("PrimaryRiskState").isin(ALLOWED_TERRITORY_STATES), F.col("Territory")).otherwise(F.lit(None)),
        )
        .drop("ZIP", "StateAbbrev", "Territory")
    )


stat_df = spark.table(ADDRESS_TABLE)
stat_df = apply_stat83_coverage_rules(stat_df)
stat_df = apply_aslob_manual_updates(stat_df)
stat_df = apply_zip_corrections(stat_df)
stat_df = attach_territory_code(stat_df, TERRITORY_FILE)
log("STAT rules, ZIP corrections and territory mapping defined (lazy).")

## 11. PIP indicator, claim code and option rules

In [ ]:
# HasPIPVUWCCoveredInd rules.
# Each rule is (coverage codes, rating basis values, indicator). Rules are
# evaluated in order, the first matching rule wins.
RATING_BASIS_RULES = [
    (["MED EXP", "INC LOSS", "ACC DEATH", "FUN EXP"], [3], 1),
    (["MED EXP", "ACC DEATH", "INC LOSS", "FUN EXP"], [0, 1, 2, 4], 0),
    (["COMB-FPB"], [3], 1),
    (["COMB-FPB"], [1, 2, 4], 0),
    (["EMB"], [1, 3], 1),
    (["EMB"], [2, 4], 0),
]

# Coverage and extension pairs that default to 0 when no indicator was derived.
PIP_DEFAULT_ZERO_PAIRS = [
    "MED EXP_NF", "MED EXP_FPB", "COMB-FPB_NF", "COMB-FPB_FPB", "EMB_NF", "EMB_FPB",
    "BROAD PIP_NF", "BROAD PIP_FPB", "ACC DEATH_NF", "ACC DEATH_FPB", "FUN EXP_NF",
    "FUN EXP_FPB", "INC LOSS_NF", "INC LOSS_FPB",
]


def derive_pip_indicator_from_rating_basis(df: DataFrame) -> DataFrame:
    """Derive HasPIPVUWCCoveredInd for IBMi Pennsylvania PIP coverages.

    The rating basis answer (policy question RatingBasis) of the risk item decides
    whether work compensation covers the PIP benefit.

    Args:
        df: Data after territory mapping.

    Returns:
        Data with HasPIPVUWCCoveredInd (NULL when no rule applies).
    """
    valid_keys = (
        df.filter(
            (F.col("PolicySource") == "IBMi")
            & (F.col("PrimaryRiskState") == "PA")
            & (F.col("CoverageExtension").isin("NF", "FPB"))
            & (F.col("RiskItemId").isNotNull())
        )
        .select("InTransactionId", "RiskItemId")
        .distinct()
    )

    rating_basis = (
        spark.table(f"{ODS_SCHEMA}.policy_question_answer")
        .filter(F.col("QuestionCode") == "RatingBasis")
        .select(
            F.col("InTransactionID").alias("InTransactionId"),
            F.col("RiskItemID").alias("RiskItemId"),
            F.col("TextValue").cast("int").alias("RB_TextValue"),
        )
    )
    valid_rating_basis = valid_keys.join(rating_basis, ["InTransactionId", "RiskItemId"], "inner")

    with_basis = (
        df.join(valid_rating_basis, ["InTransactionId", "RiskItemId"], "left")
        .select(df["*"], F.col("RB_TextValue"))
    )

    expression = None
    for codes, basis_values, indicator in RATING_BASIS_RULES:
        condition = F.col("CoverageCode").isin(*codes) & F.col("RB_TextValue").isin(*basis_values)
        expression = F.when(condition, indicator) if expression is None else expression.when(condition, indicator)
    expression = expression.when(F.col("CoverageCode") == "BROAD PIP", 0).otherwise(F.lit(None))

    return with_basis.withColumn("HasPIPVUWCCoveredInd", expression).drop("RB_TextValue")


def apply_pip_indicator_lookup(df: DataFrame) -> DataFrame:
    """Override HasPIPVUWCCoveredInd with the manual lookup where one exists."""
    lookup = read_lookup_csv(LOOKUP_FILES["has_pip_vuwc"]).select(
        "PolicyNumber", "PolicyVersion", "InTransactionId", "RiskItemId", "CoverageExtension",
        F.col("HasPIPVUWCCoveredInd").alias("HasPIPVUWCCoveredInd_new"),
    )
    joined = (
        df.alias("f")
        .join(
            lookup.alias("h"),
            [
                F.col("f.PolicyNumber") == F.col("h.PolicyNumber"),
                F.col("f.PolicyVersion") == F.col("h.PolicyVersion"),
                F.col("f.InTransactionId") == F.col("h.InTransactionId"),
                F.col("f.RiskItemId") == F.col("h.RiskItemId"),
                F.col("f.CoverageExtension") == F.col("h.CoverageExtension"),
            ],
            "left",
        )
        .select(F.col("f.*"), F.col("h.HasPIPVUWCCoveredInd_new"))
    )
    return joined.withColumn(
        "HasPIPVUWCCoveredInd",
        F.coalesce(F.col("HasPIPVUWCCoveredInd_new"), F.col("HasPIPVUWCCoveredInd")),
    ).drop("HasPIPVUWCCoveredInd_new")


def apply_claim_code_and_option_rules(df: DataFrame) -> DataFrame:
    """Normalize coverage extension, derive ClaimCode and Option1, default PIP indicator.

    Rules:
        1. BA PLUS with extension CBI or BI becomes CM. MED EXP with empty extension becomes NF.
        2. ClaimCode for root coverages without one is CoverageCode__CoverageExtension.
        3. UBI and UIM in Pennsylvania without Option1 get Option1 NS.
        4. PIP style pairs without an indicator default to 0.
           The helper column "pair" is kept for schema compatibility with the
           original notebook outputs.
    """
    df = df.withColumn(
        "CoverageExtension",
        F.when(
            (F.col("CoverageCode") == "BA PLUS") & F.col("CoverageExtension").isin("CBI", "BI"), "CM"
        )
        .when(
            (F.col("CoverageCode") == "MED EXP")
            & (F.col("CoverageExtension").isNull() | (F.col("CoverageExtension") == "")),
            "NF",
        )
        .otherwise(F.col("CoverageExtension")),
    )

    df = df.withColumn(
        "ClaimCode",
        F.when(
            F.col("ParentCoverageId").isNull() & F.col("ClaimCode").isNull(),
            F.concat_ws("__", F.col("CoverageCode"), F.col("CoverageExtension")),
        ).otherwise(F.col("ClaimCode")),
    )

    df = df.withColumn(
        "Option1",
        F.when(
            F.col("ClaimCode").isin("UBI__UBI", "UIM__UBI")
            & F.col("Option1").isNull()
            & (F.col("PrimaryRiskState") == "PA"),
            "NS",
        ).otherwise(F.col("Option1")),
    )

    df = df.withColumn("pair", F.concat_ws("_", F.col("CoverageCode"), F.col("CoverageExtension")))
    return df.withColumn(
        "HasPIPVUWCCoveredInd",
        F.when(
            F.col("pair").isin(PIP_DEFAULT_ZERO_PAIRS) & F.col("HasPIPVUWCCoveredInd").isNull(), 0
        ).otherwise(F.col("HasPIPVUWCCoveredInd")),
    )


pip_df = derive_pip_indicator_from_rating_basis(stat_df)
pip_df = apply_pip_indicator_lookup(pip_df).dropDuplicates()
final_policy_claim_updated_df = apply_claim_code_and_option_rules(pip_df)
log("PIP indicator, claim code and option rules defined (lazy).")

## 12. NISS code mapping

In [ ]:
# IBMi uninsured and underinsured motorist coverages: (NISS coverage code, NISS class code).
IBMI_UNINSURED_NISS_CODES = {
    "UBI HIRED": (201, 9404),
    "UBI NONOWN": (201, 9408),
    "UIM HIRED": (202, 9404),
    "UIM NONOWN": (202, 9408),
}


def prepare_policy_for_niss_mapping(df: DataFrame) -> DataFrame:
    """Clean Option1 and Deductible1 so they can be compared with the mapping file.

    Option1 keeps only NI, NI&R, NS, ST. Deductible1 is kept only for OTC__CM,
    COLLISION__CL and PIP LIMIT__NF in Delaware. Zero deductibles become NULL.
    """
    claim_code = F.upper(F.trim(F.col("ClaimCode")))
    keep_deductible = (
        (claim_code == "OTC__CM")
        | (claim_code == "COLLISION__CL")
        | ((claim_code == "PIP LIMIT__NF") & (F.upper(F.trim(F.col("PrimaryRiskState"))) == "DE"))
    )
    return (
        df.withColumn(
            "Option1_clean",
            F.when(F.col("Option1").isin("NI", "NI&R", "NS", "ST"), F.col("Option1")).otherwise(None),
        )
        .withColumn("Deductible1", F.when(~keep_deductible, None).otherwise(F.col("Deductible1")))
        .withColumn(
            "Deductible1_clean",
            F.when(F.col("Deductible1") == 0.0, None).otherwise(F.col("Deductible1")),
        )
    )


def load_niss_column_mapping() -> DataFrame:
    """Load and de-duplicate the NISS column mapping file.

    One mapping row is kept per effective coverage code, state, extension,
    deductible, option, worker compensation flag and claim code.
    """
    mapping = read_lookup_csv(LOOKUP_FILES["niss_column_mapping"])
    mapping = mapping.withColumn(
        "Deductible", F.when(F.col("Deductible") == 0, None).otherwise(F.col("Deductible"))
    )
    mapping = mapping.withColumn(
        "NISS Coverage Code", F.lpad(F.col("NISS Coverage Code").cast("string"), 3, "0")
    )
    mapping = mapping.withColumn(
        "EffectiveCoverageCode",
        F.coalesce(F.col("ODS Child Coverage Code"), F.col("Coverage Code")),
    )
    window_spec = Window.partitionBy(
        "EffectiveCoverageCode", "State", "Coverage Extension", "Deductible",
        "Deductible Applies to Option1", "Covered By Worker's Compensation", "ClaimCode",
    ).orderBy(F.lit(1))
    return (
        mapping.withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )


def map_niss_codes(policy_df: DataFrame, mapping_df: DataFrame) -> DataFrame:
    """Attach NISS coverage, subline, type of loss and class codes through the ClaimCode.

    Only rows with a ClaimCode are mapped. This matches the original notebook,
    where the result of the coverage code based (parent) join was never used.

    Args:
        policy_df: Cleaned policy and claim data.
        mapping_df: De-duplicated NISS column mapping.

    Returns:
        Rows with NISS_Coverage_Code, NISS_Subline_Code, NISS_Type_of_Loss and NISS_Class_Codes.
    """
    option_match = (
        (F.col("fp.Option1_clean") == F.col("mp.Deductible Applies to Option1"))
        | (F.col("fp.Option1_clean").isNull() & F.col("mp.Deductible Applies to Option1").isNull())
    )
    deductible_match = (
        (F.col("fp.Deductible1_clean") == F.col("mp.Deductible"))
        | (F.col("fp.Deductible1_clean").isNull() & F.col("mp.Deductible").isNull())
    )
    worker_match = (
        (F.col("fp.HasPIPVUWCCoveredInd").cast("int") == F.col("mp.Covered By Worker's Compensation").cast("int"))
        | (F.col("fp.HasPIPVUWCCoveredInd").isNull() & F.col("mp.Covered By Worker's Compensation").isNull())
    )

    child_rows = policy_df.filter(F.col("ClaimCode").isNotNull())
    return (
        child_rows.alias("fp")
        .join(
            mapping_df.alias("mp"),
            (F.col("fp.ClaimCode") == F.col("mp.ClaimCode"))
            & (F.col("fp.PrimaryRiskState") == F.col("mp.State"))
            & option_match
            & deductible_match
            & worker_match,
            "left",
        )
        .select(
            "fp.*",
            F.col("mp.NISS Coverage Code").alias("NISS_Coverage_Code"),
            F.col("mp.NISS Subline Code (1)").alias("NISS_Subline_Code"),
            F.col("mp.NISS Type of Loss (1)").alias("NISS_Type_of_Loss"),
            F.col("mp.NISS Class Codes (6)").alias("NISS_Class_Codes"),
        )
    )


def apply_niss_code_overrides(df: DataFrame) -> DataFrame:
    """Apply IBMi uninsured and BA PLUS overrides to the NISS codes."""
    is_ibmi = F.col("PolicySource") == "IBMi"
    coverage_code = F.col("CoverageCode")
    uninsured_codes = list(IBMI_UNINSURED_NISS_CODES.keys())

    df = (
        df
        .withColumn(
            "NISS_Coverage_Code",
            case_when_map(
                coverage_code,
                {k: v[0] for k, v in IBMI_UNINSURED_NISS_CODES.items()},
                is_ibmi,
                F.col("NISS_Coverage_Code"),
            ),
        )
        .withColumn(
            "NISS_Subline_Code",
            F.when(is_ibmi & coverage_code.isin(*uninsured_codes), 0).otherwise(F.col("NISS_Subline_Code")),
        )
        .withColumn(
            "NISS_Type_of_Loss",
            F.when(is_ibmi & coverage_code.isin(*uninsured_codes), 0).otherwise(F.col("NISS_Type_of_Loss")),
        )
        .withColumn(
            "NISS_Class_Codes",
            case_when_map(
                coverage_code,
                {k: v[1] for k, v in IBMI_UNINSURED_NISS_CODES.items()},
                is_ibmi,
                F.col("NISS_Class_Codes"),
            ),
        )
    )

    # BA PLUS claim codes whose extension does not exist in the mapping file.
    ba_plus_mismatch = F.col("ClaimCode").isin("BA PLUS__CBI", "BA PLUS__BI")
    return (
        df
        .withColumn("NISS_Coverage_Code", F.when(ba_plus_mismatch, F.lit(19)).otherwise(F.col("NISS_Coverage_Code")))
        .withColumn("NISS_Subline_Code", F.when(ba_plus_mismatch, F.lit(0)).otherwise(F.col("NISS_Subline_Code")))
        .withColumn("NISS_Type_of_Loss", F.when(ba_plus_mismatch, F.lit(None)).otherwise(F.col("NISS_Type_of_Loss")))
        .withColumn("NISS_Class_Codes", F.when(ba_plus_mismatch, F.lit(9999)).otherwise(F.col("NISS_Class_Codes")))
    )


def finalize_niss_class_codes(df: DataFrame) -> DataFrame:
    """Resolve the vehicle class placeholder, trim to 4 digits and apply corrected class codes."""
    df = df.withColumn(
        "NISS_Class_Codes",
        F.when(F.col("NISS_Class_Codes") == "Vehicle Class Code", F.col("ClassCode")).otherwise(F.col("NISS_Class_Codes")),
    )
    df = df.withColumn("NISS_Class_Codes", F.substring(F.col("NISS_Class_Codes"), 1, 4))

    corrected = read_lookup_csv(LOOKUP_FILES["corrected_class_code"]).select(
        F.col("Class").alias("cc_Class"), F.col("NISS Code").alias("cc_NISS_Code")
    )
    return (
        df.alias("f")
        .join(corrected.alias("c"), F.col("f.NISS_Class_Codes") == F.col("c.cc_Class"), "left")
        .withColumn(
            "NISS_Class_Codes",
            F.when(F.col("c.cc_NISS_Code").isNotNull(), F.col("c.cc_NISS_Code")).otherwise(F.col("f.NISS_Class_Codes")),
        )
        .drop("cc_Class", "cc_NISS_Code")
    )


def apply_type_of_loss_from_cause(df: DataFrame) -> DataFrame:
    """Fill NISS_Type_of_Loss from the cause of loss for claim rows; clear it for policy rows."""
    cause_mapping = (
        read_lookup_csv(LOOKUP_FILES["niss_cause_of_loss"])
        .select(
            F.col("Numeric Code").alias("map_Numeric_Code"),
            F.col("Cause Of Loss Code3").alias("map_Cause_Of_Loss_Code"),
        )
        .dropDuplicates(["map_Numeric_Code"])
    )
    joined = df.join(cause_mapping, df["CauseOfLoss"] == cause_mapping["map_Numeric_Code"], "left")
    updated = joined.withColumn(
        "NISS_Type_of_Loss",
        F.when(
            F.col("ClaimNumber").isNotNull()
            & F.col("NISS_Type_of_Loss").isNull()
            & F.col("map_Cause_Of_Loss_Code").isNotNull(),
            F.col("map_Cause_Of_Loss_Code"),
        ).otherwise(F.col("NISS_Type_of_Loss")),
    ).drop("map_Numeric_Code", "map_Cause_Of_Loss_Code")
    return updated.withColumn(
        "NISS_Type_of_Loss",
        F.when(F.col("ClaimNumber").isNull(), F.lit(None)).otherwise(F.col("NISS_Type_of_Loss")),
    )


niss_input_df = prepare_policy_for_niss_mapping(final_policy_claim_updated_df)
niss_mapped_df = map_niss_codes(niss_input_df, load_niss_column_mapping())
niss_mapped_df = apply_niss_code_overrides(niss_mapped_df)
final_enriched_df = apply_type_of_loss_from_cause(finalize_niss_class_codes(niss_mapped_df))
log("NISS coverage, subline, type of loss and class code mapping defined (lazy).")

if RUN_VALIDATIONS:
    unmapped = final_enriched_df.filter(F.col("NISS_Coverage_Code").isNull()).count()
    log(f"Rows without a NISS coverage code after mapping: {unmapped}")

## 13. Exception codes

In [ ]:
EXCEPTION_COVERAGE_CODES = ["MED EXP", "ACC DEATH", "INC LOSS", "FUN EXP"]


def limit_matches(source_column: Column, mapping_column: Column) -> Column:
    """Match a policy limit with a mapping limit that may be a number, Any or AO (all other)."""
    return (
        (mapping_column == "Any")
        | (mapping_column == "AO")
        | (mapping_column == source_column.cast("string"))
        | (mapping_column.cast("int") == source_column)
    )


def build_pa_pip_limit_pivot(df: DataFrame) -> DataFrame:
    """Pivot the PIP limits of Pennsylvania MED EXP risk items to one row per risk item.

    Args:
        df: Data after NISS code mapping.

    Returns:
        One row per (InTransactionId, RiskItemId) with columns such as
        "MED EXP Limit1" and "INC LOSS Limit2".
    """
    coverage_info = spark.table(f"{ODS_SCHEMA}.policy_coverage_info")

    med_exp_keys = (
        df.filter(
            (F.col("CoverageCode") == "MED EXP")
            & F.col("CoverageExtension").isin("NF", "FPB")
            & (F.col("PrimaryRiskState") == "PA")
        )
        .select("InTransactionId", "RiskItemId", "CoverageId")
        .dropDuplicates()
    )

    risk_item_coverages = (
        coverage_info.alias("c")
        .join(
            med_exp_keys.alias("m"),
            (F.col("c.InTransactionId") == F.col("m.InTransactionId"))
            & (F.col("c.RiskItemId") == F.col("m.RiskItemId")),
            "inner",
        )
        .filter(F.col("c.CoverageCode").isin(EXCEPTION_COVERAGE_CODES))
        .select(
            F.col("c.InTransactionId"), F.col("c.RiskItemId"), F.col("c.CoverageCode"),
            F.col("c.CoverageExtension"), F.col("c.Limit1"), F.col("c.Limit2"),
            F.col("c.CoverageId"), F.col("c.ParentCoverageId"),
        )
    )

    hierarchy = (
        coverage_info.alias("c")
        .join(
            med_exp_keys.select("InTransactionId", "CoverageId").alias("m"),
            (F.col("c.InTransactionId") == F.col("m.InTransactionId"))
            & (
                (F.col("c.CoverageId") == F.col("m.CoverageId"))
                | (F.col("c.ParentCoverageId") == F.col("m.CoverageId"))
            ),
            "inner",
        )
        .filter(F.col("c.CoverageCode").isin(EXCEPTION_COVERAGE_CODES))
        .select(F.col("c.InTransactionId"), F.col("c.CoverageCode"), F.col("c.OnsetAmount"), F.col("c.OffsetAmount"))
    )

    full_limits = (
        risk_item_coverages.alias("r")
        .join(
            hierarchy.alias("h"),
            (F.col("r.InTransactionId") == F.col("h.InTransactionId"))
            & (F.col("r.CoverageCode") == F.col("h.CoverageCode")),
            "left",
        )
        .select(
            F.col("r.InTransactionId").alias("InTransactionId"),
            F.col("r.RiskItemId").alias("RiskItemId"),
            F.col("r.CoverageCode"), F.col("r.CoverageExtension"),
            F.col("r.Limit1"), F.col("r.Limit2"), F.col("r.CoverageId"), F.col("r.ParentCoverageId"),
            F.col("h.OnsetAmount"), F.col("h.OffsetAmount"),
        )
    )

    pivot = (
        full_limits.groupBy("InTransactionId", "RiskItemId")
        .pivot("CoverageCode", EXCEPTION_COVERAGE_CODES)
        .agg(F.first("Limit1").alias("Limit1"), F.first("Limit2").alias("Limit2"))
    )
    return (
        pivot
        .withColumnRenamed("MED EXP_Limit1", "MED EXP Limit1")
        .withColumnRenamed("ACC DEATH_Limit1", "ACC DEATH Limit1")
        .withColumnRenamed("FUN EXP_Limit1", "FUN EXP Limit1")
        .withColumnRenamed("INC LOSS_Limit1", "INC LOSS Limit1")
        .withColumnRenamed("INC LOSS_Limit2", "INC LOSS Limit2")
    )


def load_exception_code_mapping() -> DataFrame:
    """Load the exception code mapping with Any and All Other normalized to Any and AO."""
    mapping = read_lookup_csv(LOOKUP_FILES["exception_code_mapping"])
    for column in ["MED EXP Limit1", "ACC DEATH Limit1", "FUN EXP Limit1", "INC LOSS Limit1", "INC LOSS Limit2"]:
        mapping = mapping.withColumn(
            column,
            F.when(F.col(column) == "Any", "Any")
            .when(F.col(column) == "All Other", "AO")
            .otherwise(F.col(column).cast("int")),
        )
    return mapping


def match_exception_codes(limit_pivot: DataFrame, mapping: DataFrame) -> DataFrame:
    """Find the NISS exception code for each risk item by matching its PIP limits.

    One code is kept per risk item.
    """
    source = limit_pivot
    for column in ["MED EXP Limit1", "ACC DEATH Limit1", "FUN EXP Limit1", "INC LOSS Limit1", "INC LOSS Limit2"]:
        source = source.withColumn(column, F.col(column).cast("int"))

    return (
        source.alias("s")
        .join(
            mapping.alias("m"),
            limit_matches(F.col("s.`MED EXP Limit1`"), F.col("m.`MED EXP Limit1`"))
            & limit_matches(F.col("s.`ACC DEATH Limit1`"), F.col("m.`ACC DEATH Limit1`"))
            & limit_matches(F.col("s.`FUN EXP Limit1`"), F.col("m.`FUN EXP Limit1`"))
            & (
                limit_matches(F.col("s.`INC LOSS Limit1`"), F.col("m.`INC LOSS Limit1`"))
                | limit_matches(F.col("s.`INC LOSS Limit2`"), F.col("m.`INC LOSS Limit2`"))
            ),
            "left",
        )
        .select("s.InTransactionId", "s.RiskItemId", F.col("m.`NISS Exception Code`").alias("NISS_Exception_Code"))
        .dropDuplicates(["InTransactionId", "RiskItemId"])
    )


def apply_exception_code_rules(df: DataFrame, matched_codes: DataFrame) -> DataFrame:
    """Attach exception codes and apply the EMB and COMB-FPB limit rules.

    Exception codes are kept only for COMB-FPB, EMB, MED EXP, ACC DEATH, FUN EXP
    and INC LOSS with extension NF.
    """
    df = df.alias("f").join(matched_codes.alias("x"), ["InTransactionId", "RiskItemId"], "left")

    limit_rule = F.when(
        F.col("CoverageCode") == "EMB",
        F.when(F.col("Limit1") == 100000, "01")
        .when(F.col("Limit1") == 300000, "02")
        .when(F.col("Limit1") == 500000, "03")
        .when(F.col("Limit1") == 1000000, "04"),
    ).when(
        F.col("CoverageCode") == "COMB-FPB",
        F.when(F.col("Limit1") == 12500, "06")
        .when(F.col("Limit1") == 17500, "01")
        .when(F.col("Limit1") == 50000, "02")
        .when(F.col("Limit1") == 100000, "03")
        .when(F.col("Limit1") == 177500, "07")
        .when(F.col("Limit1") == 200000, "04")
        .when(F.col("Limit1") == 277500, "05")
        .otherwise("09"),
    )
    df = df.withColumn("NISS_Exception_Code", limit_rule.otherwise(F.col("NISS_Exception_Code")))
    df = df.withColumn("NISS_Exception_Code", F.lpad(F.col("NISS_Exception_Code").cast("string"), 2, "0"))

    keep_code = (
        F.col("CoverageCode").isin("COMB-FPB", "EMB", "MED EXP", "ACC Death", "FUN EXP")
        | ((F.col("CoverageCode") == "INC LOSS") & (F.col("CoverageExtension") == "NF"))
    )
    return df.withColumn("NISS_Exception_Code", F.when(keep_code, F.col("NISS_Exception_Code")).otherwise(None))


exception_matches = match_exception_codes(build_pa_pip_limit_pivot(final_enriched_df), load_exception_code_mapping())
with_exception_df = apply_exception_code_rules(final_enriched_df, exception_matches)
log("Exception code mapping defined (lazy).")

## 14. Exposure, state codes and data fixes

Saved to `Final_Transformed_Data`.

In [ ]:
def apply_csv_data_fix(df: DataFrame, fix_df: DataFrame, join_columns: List[str], claim_rows_only: bool) -> DataFrame:
    """Overwrite columns of matching rows with the values of a data fix lookup.

    Only non-NULL lookup values overwrite. Rows without a match are unchanged.

    Args:
        df: Data to fix.
        fix_df: Lookup with the join columns plus the columns to update.
        join_columns: Columns that identify the rows to fix.
        claim_rows_only: When True, only rows that have a ClaimNumber are updated.

    Returns:
        Data with the same columns as df.
    """
    joined = df.alias("f2").join(fix_df.alias("f1"), on=join_columns, how="left")
    update_columns = [c for c in fix_df.columns if c not in join_columns]

    select_columns = []
    for column in df.columns:
        if column in update_columns:
            condition = F.col(f"f1.{column}").isNotNull()
            if claim_rows_only:
                condition = F.col("f2.ClaimNumber").isNotNull() & condition
            select_columns.append(F.when(condition, F.col(f"f1.{column}")).otherwise(F.col(f"f2.{column}")).alias(column))
        else:
            select_columns.append(F.col(f"f2.{column}"))
    return joined.select(*select_columns)


def apply_exposure_rules(df: DataFrame) -> DataFrame:
    """Calculate Final_Exposure for policy rows.

    Rules:
        1. Origami policy rows without a ProrataPercent use 1.
        2. Raw exposure is ProrataPercent multiplied by 12 (months).
        3. Round half up to a whole number.
        4. The sign follows the premium: negative premium gives negative exposure.
    These rules apply per booked transaction, so they are valid for any period.
    """
    df = df.withColumn(
        "ProrataPercent",
        F.when(
            (F.col("PolicySource") == "Origami")
            & F.col("ClaimNumber").isNull()
            & (
                F.col("ProrataPercent").isNull()
                | (F.trim(F.col("ProrataPercent")) == "")
                | (F.col("ProrataPercent") == "null")
            ),
            F.lit(1),
        ).otherwise(F.col("ProrataPercent")),
    )

    return (
        df
        .withColumn("Premium", F.when(F.col("OnSetAmount") != 0, F.col("OnSetAmount")).otherwise(F.col("OffsetAmount")))
        .withColumn("Raw_Exposure", F.col("ProrataPercent") * 12)
        .withColumn(
            "Rounded_Exposure",
            F.when(
                (F.col("Raw_Exposure") - F.floor(F.col("Raw_Exposure"))) >= 0.5,
                F.ceil(F.col("Raw_Exposure")),
            ).otherwise(F.floor(F.col("Raw_Exposure"))),
        )
        .withColumn(
            "Final_Exposure",
            F.when(F.col("Premium") < 0, -F.col("Rounded_Exposure")).otherwise(F.col("Rounded_Exposure")),
        )
        .drop("Premium", "Raw_Exposure", "Rounded_Exposure")
    )


def apply_state_code_mapping(df: DataFrame) -> DataFrame:
    """Replace the state abbreviation in PrimaryRiskState with the NISS state code."""
    return df.withColumn(
        "PrimaryRiskState",
        case_when_map(F.col("PrimaryRiskState"), NISS_STATE_CODES, F.lit(True), F.col("PrimaryRiskState")),
    )


def finalize_transformed_data(df: DataFrame) -> DataFrame:
    """Apply exposure, state mapping, override registers and CSV data fixes.

    Order matters: the state mapping runs before the CSV data fix because the
    data fix file identifies rows by the NISS state code.
    """
    df = apply_exposure_rules(df)
    df = apply_state_code_mapping(df)
    df = apply_override_rules(df, "CoverageCode", CLAIM_COVERAGE_CODE_OVERRIDES)

    offset_fix = read_lookup_csv(LOOKUP_FILES["datafix_offset"])
    df = apply_csv_data_fix(
        df, offset_fix,
        ["InTransactionId", "PolicyNumber", "PolicyVersion", "PrimaryRiskState", "CoverageCode", "OffsetAmount"],
        claim_rows_only=False,
    )

    claim_fix = read_lookup_csv(LOOKUP_FILES["datafix_claims"])
    df = apply_csv_data_fix(df, claim_fix, ["ClaimNumber", "InTransactionId", "CoverageCode"], claim_rows_only=True)

    return apply_override_rules(df, "NISS_Exception_Code", EXCEPTION_CODE_OVERRIDES)


transformed_df = finalize_transformed_data(with_exception_df)

FINAL_TABLE = ctx.table_name("Final_Transformed_Data")
write_delta_table(transformed_df, FINAL_TABLE)
log(f"Saved {FINAL_TABLE}")

## 15. NISS record layout, Transmittal file and Summary file

In [ ]:
# Fixed width blank fields: (column name, length). The values are NULL padded
# strings, exactly as in the original layout code.
BLANK_FIELDS = [
    ("Blank_1", 1), ("Blank_5_6", 2), ("Blank_14_15", 2), ("Blank_18_22", 5), ("Blank_26_30", 5),
    ("RatingZoneCode", 1), ("TerminalZoneCode", 2), ("Blank_42_45", 4), ("EligibilityPointsCode", 2),
    ("Blank_54_55", 2), ("AgeGroupCode", 1), ("ManufacturersModelYear", 2), ("Blank_59_60", 2),
    ("ForgivenessCode", 1), ("PassiveRestraintCode", 2), ("DefensiveDriverCreditCode", 1),
    ("Blank_69", 1), ("AntiTheftDeviceCode", 1), ("DaytimeRunningLampsDiscountCode", 1),
    ("Blank_72_74", 3), ("PolicyLimitsCode", 2), ("DeductibleCode", 2), ("Blank_79", 1),
    ("SupplementalSpousalLiabilityCode", 1), ("Blank_81", 1), ("Blank_83", 1),
    ("LiabilityNoFaultCode", 1), ("Blank_86_98", 13), ("Blank_102_103", 2), ("ReservedForNISSUse", 8),
]

# Constant fields taken from the Constant table: (column name, length).
CONSTANT_FIELDS = [
    ("CompanyNumber", 3), ("LineCode", 1), ("CommercialIndicatorCode", 1), ("ClaimantLevelIndicator", 1),
]

# Classification codes for which WrittenExposure is not reported.
CLASS_CODES_WITHOUT_EXPOSURE = [
    "608000", "604000", "580000", "600100", "600300", "600500", "600900", "648000", "649000",
    "780000", "790000", "708000", "940900", "949500", "940600", "940500", "940400", "999900",
    "940800", "941100", "941400", "941200", "946400",
]

# Order of the 55 columns required by NISS.
NISS_RECORD_COLUMNS = [
    "Blank_1", "CompanyNumber", "Blank_5_6", "CalendarYear", "CallYear", "StateCode",
    "LineCode", "Blank_14_15", "AccidentYear", "Blank_18_22", "NISSCoverageCode",
    "Blank_26_30", "TerritoryCode", "RatingZoneCode", "TerminalZoneCode", "NSSZIPCode",
    "Blank_42_45", "ClassificationCode", "EligibilityPointsCode", "Blank_54_55",
    "AgeGroupCode", "ManufacturersModelYear", "Blank_59_60", "CommercialIndicatorCode",
    "NISS_Exception_Code", "ForgivenessCode", "ClaimantLevelIndicator", "PassiveRestraintCode",
    "DefensiveDriverCreditCode", "Blank_69", "AntiTheftDeviceCode",
    "DaytimeRunningLampsDiscountCode", "Blank_72_74", "PolicyLimitsCode", "DeductibleCode",
    "Blank_79", "SupplementalSpousalLiabilityCode", "Blank_81", "SublineOfBusinessCode",
    "Blank_83", "TypeOfLossCode", "LiabilityNoFaultCode", "Blank_86_98",
    "AnnualStatementLineOfBusinessCode", "Blank_102_103", "WrittenExposure",
    "WrittenPremium", "PaidLosses", "PaidAllocatedLossAdjustmentExpenses",
    "OutstandingLossesIncludingALAE", "NumberOfPaidClaims",
    "NumberOfOutstandingClaims", "ReservedForNISSUse", "ClaimNumberPremiumID",
    "ClaimantNumber",
]


def load_niss_constants() -> Dict[str, str]:
    """Return the NISS constants, seeding missing keys into the Constant table.

    Existing rows are never overwritten, so values edited in the table are
    respected. Re-running the notebook does not add duplicate rows.
    """
    spark.sql(f"CREATE TABLE IF NOT EXISTS {CONSTANT_TABLE} (ColumnName STRING, ConstantValue STRING) USING DELTA")
    seed = spark.createDataFrame(list(NISS_CONSTANT_DEFAULTS.items()), "ColumnName STRING, ConstantValue STRING")
    seed.createOrReplaceTempView("niss_constant_seed")
    spark.sql(f"""
        MERGE INTO {CONSTANT_TABLE} t
        USING niss_constant_seed s
        ON t.ColumnName = s.ColumnName
        WHEN NOT MATCHED THEN INSERT (ColumnName, ConstantValue) VALUES (s.ColumnName, s.ConstantValue)
    """)
    return {row["ColumnName"]: row["ConstantValue"] for row in spark.table(CONSTANT_TABLE).collect()}


def build_niss_record_fields(df: DataFrame, constants: Dict[str, str], period: ReportingPeriod) -> DataFrame:
    """Create the NISS record fields (positions 1 to 55) with their fixed widths.

    CalendarYear and CallYear come from the reporting period. Calendar year is the
    year of the period and call year is calendar year plus one.

    Args:
        df: Final transformed data.
        constants: Output of load_niss_constants.
        period: Reporting period.

    Returns:
        Data with all NISS record columns added.
    """
    calendar_year_2 = str(period.calendar_year)[-2:]
    call_year_2 = str(period.call_year)[-2:]

    for name, length in BLANK_FIELDS:
        df = df.withColumn(name, F.rpad(F.lit(None).cast("string"), length, " "))
    for name, length in CONSTANT_FIELDS:
        df = df.withColumn(name, F.rpad(F.lit(constants[name]), length, " "))

    df = (
        df
        .withColumn("CalendarYear", F.lpad(F.lit(calendar_year_2), 2, " "))
        .withColumn("CallYear", F.lpad(F.lpad(F.lit(call_year_2), 2, "0"), 2, " "))
        .withColumn("StateCode", F.rpad(F.col("PrimaryRiskState").cast("string"), 2, " "))
        .withColumn("AccidentYear", F.substring(F.col("AccidentYear").cast("string"), 3, 2))
        .withColumn("NISSCoverageCode", F.lpad(F.col("NISS_Coverage_Code").cast("string"), 3, "0"))
        .withColumn("TerritoryCode", F.lpad(F.col("TerritoryCode").cast("string"), 3, "0"))
        .withColumn("NSSZIPCode", F.lit(None).cast("string"))
        .withColumn("ClassificationCode", F.rpad(F.col("NISS_Class_Codes").cast("string"), 6, "0"))
        .withColumn("NISS_Exception_Code", F.lpad(F.col("NISS_Exception_Code").cast("string"), 2, "0"))
        .withColumn("SublineOfBusinessCode", F.rpad(F.col("NISS_Subline_Code").cast("string"), 1, " "))
        .withColumn("TypeOfLossCode", F.rpad(F.col("NISS_Type_of_Loss").cast("string"), 1, " "))
        .withColumn("AnnualStatementLineOfBusinessCode", F.rpad(F.col("ASLOB").cast("string"), 3, " "))
        .withColumn("WrittenExposure", F.col("Final_Exposure"))
    )

    # Written premium: onset amount when present, otherwise offset amount, whole dollars.
    premium = (
        F.when(F.col("OnsetAmount").isNotNull() & (F.col("OnsetAmount") != 0), F.col("OnsetAmount"))
        .when(F.col("OffsetAmount").isNotNull() & (F.col("OffsetAmount") != 0), F.col("OffsetAmount"))
    )
    df = df.withColumn("WrittenPremium", F.regexp_replace(premium.cast("string"), ",", "").cast("double"))
    df = df.withColumn("WrittenPremium", F.round(F.col("WrittenPremium"), 0).cast("int"))

    df = (
        df
        .withColumn("PaidLosses", F.round(F.col("PaidLossAmount").cast("double"), 0).cast("int"))
        .withColumn("PaidAllocatedLossAdjustmentExpenses", F.round(F.col("AllocatedLossAdjExp").cast("double"), 0).cast("int"))
        .withColumn("OutstandingLossesIncludingALAE", F.round(F.col("ReserveAmount").cast("double"), 0).cast("int"))
        .withColumn("NumberOfPaidClaims", F.col("PaidCount").cast("int"))
        .withColumn("NumberOfOutstandingClaims", F.col("ReserveCount").cast("int"))
    )

    # ClaimNumberPremiumID uses the raw ClaimantNumber, so it is built before ClaimantNumber is padded.
    premium_id = F.when(
        F.col("ClaimNumber").isNotNull(),
        F.concat_ws("_", F.col("ClaimNumber").cast("string"), F.col("ClaimantNumber").cast("string"), F.col("CoverageCode").cast("string")),
    ).otherwise(
        F.concat_ws("_", F.col("InTransactionId").cast("string"), F.col("CoverageId").cast("string"))
    )
    df = df.withColumn("ClaimNumberPremiumID", F.rpad(premium_id.substr(1, 16), 16, " "))

    return df.withColumn(
        "ClaimantNumber", F.lpad(F.col("ClaimantNumber").cast("string").substr(1, 3), 3, "0")
    )


def apply_record_adjustments(df: DataFrame) -> DataFrame:
    """Apply the final NISS reporting rules to the record fields.

    Rules:
        1. CommercialIndicatorCode is kept only for classification codes listed in
           the ClassesWithClaimInd lookup.
        2. WrittenExposure is blank for the classification codes without exposure.
        3. Pennsylvania BROAD PIP always uses exception code 49.
    """
    claim_indicator_classes = (
        read_lookup_csv(LOOKUP_FILES["classes_with_claim_ind"])
        .withColumn(
            "Class Codes - first 3 characters",
            F.lpad(F.col("Class Codes - first 3 characters").cast("string"), 3, "0"),
        )
    )
    valid_codes = [row[0] for row in claim_indicator_classes.select("Class Codes - first 3 characters").distinct().collect()]

    df = df.withColumn(
        "CommercialIndicatorCode",
        F.when(F.substring(F.col("ClassificationCode"), 1, 3).isin(valid_codes), F.col("CommercialIndicatorCode")).otherwise(None),
    )
    df = df.withColumn(
        "WrittenExposure",
        F.when(F.col("ClassificationCode").isin(CLASS_CODES_WITHOUT_EXPOSURE), None).otherwise(F.col("WrittenExposure")),
    )
    df = df.withColumn(
        "NISS_Exception_Code",
        F.when(
            (F.col("PrimaryRiskState") == "37") & (F.col("CoverageCode") == "BROAD PIP"), F.lit("49")
        ).otherwise(F.col("NISS_Exception_Code")),
    )
    return df.withColumn("NISS_Exception_Code", F.lpad(F.col("NISS_Exception_Code").cast("string"), 2, "0"))


def null_string(name: str) -> Column:
    """Return a NULL string column with the given name (blank record field)."""
    return F.lit(None).cast("string").alias(name)


def build_transmittal(df: DataFrame, constants: Dict[str, str], period: ReportingPeriod) -> DataFrame:
    """Build the Transmittal (T) record: one row of totals per state."""
    call_year_2 = str(period.call_year)[-2:]
    totals = df.groupBy("PrimaryRiskState").agg(
        F.sum("WrittenExposure").alias("WrittenExposure"),
        F.sum("WrittenPremium").alias("WrittenPremium"),
        F.sum("PaidLosses").alias("PaidLosses"),
        F.sum("PaidAllocatedLossAdjustmentExpenses").alias("PaidALAE"),
        F.sum("OutstandingLossesIncludingALAE").alias("OutstandingLosses"),
        F.sum("NumberOfPaidClaims").alias("NumPaidClaims"),
        F.sum("NumberOfOutstandingClaims").alias("NumOutstandingClaims"),
    )
    return totals.select(
        F.lit("T").alias("TransmittalIdentifierCode"),
        F.lit(constants["CompanyNumber"]).alias("CompanyNumber"),
        null_string("Blank_5_8"),
        F.lit(call_year_2).alias("CallYear"),
        F.col("PrimaryRiskState").alias("StateCode"),
        F.lit(constants["LineCode"]).alias("LineCode"),
        null_string("Blank_14_17"),
        null_string("RefileIndicator"),
        null_string("Blank_19"),
        null_string("NTRIndicator"),
        null_string("Blank_21_103"),
        F.col("WrittenExposure").cast("int"),
        F.col("WrittenPremium"),
        F.col("PaidLosses").cast("int"),
        F.col("PaidALAE").cast("int"),
        F.col("OutstandingLosses").cast("int"),
        F.col("NumPaidClaims").cast("int"),
        F.col("NumOutstandingClaims").cast("int"),
    )


def build_summary(record_count: int, constants: Dict[str, str], period: ReportingPeriod) -> DataFrame:
    """Build the Summary (S) record holding the data record count."""
    call_year_2 = str(period.call_year)[-2:]
    return spark.createDataFrame([(1,)], ["dummy"]).select(
        F.lit("S").alias("TransmittalIdentifierCode"),
        F.lit(constants["CompanyNumber"]).alias("CompanyNumber"),
        null_string("Blank_1"),
        F.lit(call_year_2).alias("CallYear"),
        null_string("Blank_2"),
        F.lit(constants["LineCode"]).alias("LineCode"),
        null_string("Blank_3"),
        null_string("RefileIndicator"),
        null_string("Blank_5"),
        F.lit(record_count).alias("RecordCount"),
        null_string("Blank_4"),
    )


niss_constants = load_niss_constants()
report_df = apply_record_adjustments(
    build_niss_record_fields(spark.table(FINAL_TABLE), niss_constants, ctx.period)
)
report_df = report_df.persist(StorageLevel.MEMORY_AND_DISK)

record_count = report_df.count()
log(f"NISS data records for {ctx.period.run_mode} {ctx.period.label}: {record_count}")
if record_count == 0:
    raise RuntimeError("The report contains no records. Check the period and the source data before publishing.")

# Transmittal and Summary files.
write_csv_folder(build_transmittal(report_df, niss_constants, ctx.period), ctx.output_path("TransmittalFILE"))
write_csv_folder(build_summary(record_count, niss_constants, ctx.period), ctx.output_path("SUMMARYFILE"))

# Full detail file with header (all working columns, used for reconciliation).
write_csv_folder(report_df, ctx.output_path("Pre_Final_Report_File"), header=True)

# Final NISS layout: Delta table and CSV without header.
niss_final_df = report_df.select(NISS_RECORD_COLUMNS)
NISS_REPORT_TABLE = ctx.table_name("NISS_Final_Report")
write_delta_table(niss_final_df, NISS_REPORT_TABLE)
write_csv_folder(niss_final_df, ctx.output_path("First_NISS_FORMAT_FILE"))

log(f"Saved {NISS_REPORT_TABLE} and {ctx.output_path('First_NISS_FORMAT_FILE')}")
report_df.unpersist()

elapsed = time.time() - NOTEBOOK_START_TIME
log(f"Total execution time: {elapsed:.0f} seconds ({elapsed / 60:.2f} minutes)")